# ============================================================
# 【这是最终版本，请只用这一份，之前发的其他版本可以忽略】
#
# TCM 39类中药饮片图像分类训练脚本
# 模型：YOLOv11 分类（yolo11n-cls.pt）
# 平台：Google Colab
#
# 【使用方法】
# 本文件分成7个步骤，每个步骤前面用 "# ===== 步骤N ===== " 标出
# 操作方式：
#   1. 打开 https://colab.research.google.com/，新建一个空白笔记本
#   2. 把"步骤1"下面的代码，整段复制，粘贴到第一个代码框里，点击左边的运行按钮（▶）
#   3. 等它运行完（左边圆圈转完，出现绿色勾或者打印文字），再复制"步骤2"到一个新的代码框运行
#   4. 依次做完步骤3、4、5、6
#   5. 步骤6跑完后，截图发给我看结果，我确认没问题后你再跑步骤7

In [1]:
# ===== 步骤1：连接 Google Drive =====
from google.colab import drive
drive.mount('/content/drive')
# 运行后会弹出授权窗口，点"连接到 Google Drive"或"允许"
# 成功标志：打印出 "Mounted at /content/drive"


Mounted at /content/drive


In [2]:
# ===== 步骤2：解压数据集（★这里要改文件名★）=====
import zipfile
import os
import shutil

# ↓↓↓ 把下面这一行的文件名，改成你自己上传到 Drive 里的压缩包真实名字 ↓↓↓
ZIP_PATH = "/content/drive/MyDrive/toxic_herb_dataset.zip"

EXTRACT_DIR = "/content/dataset"

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("解压完成！")
top_items = os.listdir(EXTRACT_DIR)
print("解压后顶层内容：", top_items)

EXPECTED = {"train", "val", "test", "test_subset"}
if EXPECTED.issubset(set(top_items)):
    DATA_ROOT = EXTRACT_DIR
    print(f"结构正常，DATA_ROOT = {DATA_ROOT}")
elif len(top_items) == 1:
    DATA_ROOT = os.path.join(EXTRACT_DIR, top_items[0])
    print(f"检测到多了一层文件夹，已自动修正，DATA_ROOT = {DATA_ROOT}")
else:
    DATA_ROOT = EXTRACT_DIR
    print(f"结构可能异常，请检查，当前 DATA_ROOT = {DATA_ROOT}")

解压完成！
解压后顶层内容： ['test_subset', 'test', 'val', 'train']
结构正常，DATA_ROOT = /content/dataset


In [4]:
# ===== 步骤3：检查是否为39类 =====
SPLITS = ["train", "val", "test", "test_subset"]
all_good = True

print("=== 类别数量检查 ===")
for split in SPLITS:
    split_path = os.path.join(DATA_ROOT, split)
    if not os.path.exists(split_path):
        print(f"{split}: 文件夹不存在")
        all_good = False
        continue
    folders = [f for f in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, f))]
    ok = len(folders) == 39
    print(f"{split:<15}: {len(folders)} 个类别 {'(正确)' if ok else '(数量不对！)'}")
    if not ok:
        all_good = False
        print(f"   实际文件夹: {sorted(folders)}")

print("检查通过，可以继续" if all_good else "有问题，请把上面结果发给我")

for split in SPLITS:
    cache_file = os.path.join(DATA_ROOT, f"{split}.cache")
    if os.path.exists(cache_file):
        os.remove(cache_file)

=== 类别数量检查 ===
train          : 39 个类别 (正确)
val            : 39 个类别 (正确)
test           : 39 个类别 (正确)
test_subset    : 39 个类别 (正确)
检查通过，可以继续


In [5]:
# ===== 步骤4：统计每类图片数量 =====
IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
train_path = os.path.join(DATA_ROOT, "train")
class_names = sorted([f for f in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, f))])

print(f"{'类别':<20}", end="")
for split in SPLITS:
    print(f"{split:>14}", end="")
print()

totals = {s: 0 for s in SPLITS}
for cls in class_names:
    print(f"{cls:<20}", end="")
    for split in SPLITS:
        folder = os.path.join(DATA_ROOT, split, cls)
        count = len([f for f in os.listdir(folder) if f.lower().endswith(IMG_EXTS)]) if os.path.exists(folder) else 0
        totals[split] += count
        print(f"{str(count):>14}", end="")
    print()

print("\n总计：")
for split in SPLITS:
    print(f"  {split}: {totals[split]} 张")

类别                           train           val          test   test_subset
badou                           60            20            20            14
baifuzi                         60            20            20             8
baiguo                          60            20            20             8
banxia                          60            20            20            13
beidougen                       51            16            16            14
caowu                           56            18            18            10
changshan                       43            12            13             8
chonglou                        59            19            20            15
chuanlianzi                     60            20            20            20
gansui                          47            15            15            11
heshi                           41            13            13            10
hongdaji                        54            17            17            15

In [6]:
# ===== 步骤5：安装工具包 =====
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.7/235.7 GB disk)


In [7]:
# ============================================================
# 【步骤6 完整合并版】清理数据 + 试训练（3轮）
# 以后每次要训练，只需要跑这一份，不用再分开跑清理脚本
# ============================================================

import os
import shutil

DATA_ROOT = "/content/dataset"
SPLITS = ["train", "val", "test", "test_subset"]

TARGET_CLASSES = set([
    "badou", "baifuzi", "baiguo", "banxia", "beidougen", "caowu", "changshan", "chonglou",
    "chuanlianzi", "gansui", "heshi", "hongdaji", "huajiao", "jili", "jiulixiang", "kulianpi",
    "langdu", "liangmianzhen", "maqianzi", "mianmaguanzhong", "mubiezi", "naoyanghua",
    "qianjinzi", "qianniuzi", "shancigu", "shanglu", "shechuangzi", "tiannanxing",
    "tianxianzi", "tujingpi", "wuzhuyu", "xiangjiapi", "xiangsizi", "xianmao", "yadanzi",
    "yangjinhua", "yingsuqiao", "yuanhua", "zhuyazao"
])

print("===== 第一步：删除所有 .ipynb_checkpoints 隐藏文件夹 =====\n")
removed_ckpt = 0
for root, dirs, files in os.walk(DATA_ROOT):
    if ".ipynb_checkpoints" in dirs:
        target = os.path.join(root, ".ipynb_checkpoints")
        shutil.rmtree(target)
        print(f"已删除: {target}")
        removed_ckpt += 1
print(f"共删除 {removed_ckpt} 个 .ipynb_checkpoints 文件夹\n")

print("===== 第二步：删除不在39类清单里的多余顶层文件夹 =====\n")
for split in SPLITS:
    split_path = os.path.join(DATA_ROOT, split)
    if not os.path.exists(split_path):
        print(f"{split}: 路径不存在，跳过")
        continue
    actual_folders = set(os.listdir(split_path))
    extra = actual_folders - TARGET_CLASSES
    for extra_item in extra:
        extra_path = os.path.join(split_path, extra_item)
        if os.path.isdir(extra_path):
            shutil.rmtree(extra_path)
            print(f"[{split}] 删除多余文件夹: {extra_item}")
        elif os.path.isfile(extra_path):
            os.remove(extra_path)
            print(f"[{split}] 删除多余文件: {extra_item}")

print("\n===== 第三步：清除所有旧缓存文件 =====\n")
for split in SPLITS:
    cache_file = os.path.join(DATA_ROOT, f"{split}.cache")
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"已清除缓存: {split}.cache")

print("\n===== 第四步：最终核查（用'含图片的目录数'这个严格口径）=====\n")
IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
all_ok = True
for split in SPLITS:
    split_path = os.path.join(DATA_ROOT, split)
    if not os.path.exists(split_path):
        print(f"  {split:<15}: 路径不存在")
        all_ok = False
        continue
    dirs_with_images = set()
    for r, d, files in os.walk(split_path):
        if any(f.lower().endswith(IMG_EXTS) for f in files):
            dirs_with_images.add(os.path.relpath(r, split_path))
    count = len(dirs_with_images)
    ok = count == 39
    all_ok = all_ok and ok
    print(f"  {split:<15}: {count} 个含图片的目录  {'✅' if ok else '⚠️ 仍不对'}")
    if not ok:
        print(f"    详细目录列表: {sorted(dirs_with_images)}")

if not all_ok:
    raise SystemExit("\n数据清理后仍不是39类，请检查上面详细列表，不要继续训练，把结果发给我")

print("\n数据确认干净，开始试训练（3轮）...\n")

# ===== 第五步：试训练 =====
from ultralytics import YOLO

model = YOLO("yolo11n-cls.pt")
results = model.train(
    data=DATA_ROOT,
    epochs=3,
    imgsz=224,
    batch=32,
    project="/content/drive/MyDrive/TCM_YOLOv11_runs",
    name="quick_test_final"
)

print("\n试训练完成！请检查上面日志：train/val/test 是否都显示 39 classes，且没有红色 ERROR")
print("如果没有 ERROR，把结果发给我确认后，就可以进入步骤7（正式训练50轮）")

===== 第一步：删除所有 .ipynb_checkpoints 隐藏文件夹 =====

共删除 0 个 .ipynb_checkpoints 文件夹

===== 第二步：删除不在39类清单里的多余顶层文件夹 =====


===== 第三步：清除所有旧缓存文件 =====


===== 第四步：最终核查（用'含图片的目录数'这个严格口径）=====

  train          : 39 个含图片的目录  ✅
  val            : 39 个含图片的目录  ✅
  test           : 39 个含图片的目录  ✅
  test_subset    : 39 个含图片的目录  ✅

数据确认干净，开始试训练（3轮）...

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, f

In [ ]:
# ============================================================
# 【步骤7 】训练（50轮）
# 一是patience=15表示如果验证集准确连续15轮没有提升就会提前停止，防止过度率；二是训练前记得先跑一轮import os; print(os.path.exists("/content/dataset"))确认Colab没有断连False就说明session重置了
# ==========================================================
model_full = YOLO("yolo11n-cls.pt")
results_full = model_full.train(
    data=DATA_ROOT,
    epochs=50,
    imgsz=224,
    batch=32,
    patience=15,
    project="/content/drive/MyDrive/TCM_YOLOv11_runs",
    name="full_train_baseline"
)

print("正式训练完成！结果保存在 Drive 的 TCM_YOLOv11_runs/full_train_baseline 文件夹")

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=full_train_baseline, nbs=64, nms=False, o

In [ ]:
# ============================================================
# 步骤8（修复版）：YOLOv12n-cls 主基线训练
#
# 问题根源：Ultralytics官方仓库(ultralytics/assets)只发布了YOLOv12的
# 检测权重，没有发布分类版本(-cls)的预训练权重。
# 分类版权重是YOLOv12原作者团队(sunsmarterjie)单独发布在他们自己的
# GitHub仓库release里的，需要手动指定完整下载链接。
# ============================================================

import os

if not os.path.exists("/content/dataset"):
    raise SystemExit("检测到 /content/dataset 不存在，请先重跑步骤1、2，再重跑步骤6的清理部分")

DATA_ROOT = "/content/dataset"

# 直接从原作者仓库下载官方 YOLOv12n 分类预训练权重
!wget -q "https://github.com/sunsmarterjie/yolov12/releases/download/cls/yolov12n-cls.pt" -O yolov12n-cls.pt

# 校验下载是否成功
import os
if not os.path.exists("yolov12n-cls.pt") or os.path.getsize("yolov12n-cls.pt") < 1000000:
    raise SystemExit("下载失败或文件不完整，请检查网络连接，或把报错发给我")

print(f"下载成功，文件大小: {os.path.getsize('yolov12n-cls.pt') / 1024 / 1024:.2f} MB")

from ultralytics import YOLO

model_v12n = YOLO("yolov12n-cls.pt")
results_v12n = model_v12n.train(
    data=DATA_ROOT,
    epochs=50,
    imgsz=224,
    batch=32,
    patience=15,
    project="/content/drive/MyDrive/TCM_YOLOv12_runs",
    name="v12n_baseline"
)

print("\nYOLOv12n-cls 基线训练完成！")
print("结果保存在 Drive 的 TCM_YOLOv12_runs/v12n_baseline 文件夹")
print("请把最终 val Top-1 / Top-5 准确率、最佳轮次、参数量发给我，与YOLOv11n-cls对比")

下载成功，文件大小: 5.86 MB
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov12n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v12n_baseline, nbs=64

In [ ]:
# ============================================================
# 步骤9：YOLOv12s-cls 基线训练（更大容量，验证方法在不同模型规模下的一致性）
# ============================================================

import os

if not os.path.exists("/content/dataset"):
    raise SystemExit("检测到 /content/dataset 不存在，请先重跑步骤1、2，再重跑步骤6的清理部分")

DATA_ROOT = "/content/dataset"

# 从YOLOv12原作者仓库下载官方 s 版分类预训练权重
!wget -q "https://github.com/sunsmarterjie/yolov12/releases/download/cls/yolov12s-cls.pt" -O yolov12s-cls.pt

if not os.path.exists("yolov12s-cls.pt") or os.path.getsize("yolov12s-cls.pt") < 1000000:
    raise SystemExit("下载失败或文件不完整，请检查网络连接，或把报错发给我")

print(f"下载成功，文件大小: {os.path.getsize('yolov12s-cls.pt') / 1024 / 1024:.2f} MB")

from ultralytics import YOLO

model_v12s = YOLO("yolov12s-cls.pt")
results_v12s = model_v12s.train(
    data=DATA_ROOT,
    epochs=50,
    imgsz=224,
    batch=32,
    patience=15,
    project="/content/drive/MyDrive/TCM_YOLOv12_runs",
    name="v12s_baseline"
)

print("\nYOLOv12s-cls 基线训练完成！")
print("结果保存在 Drive 的 TCM_YOLOv12_runs/v12s_baseline 文件夹")
print("请把最终 val Top-1 / Top-5 准确率、最佳轮次、参数量发给我，与 YOLOv11n-cls / YOLOv12n-cls 三方对比")

下载成功，文件大小: 14.05 MB
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov12s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v12s_baseline, nbs=6

In [ ]:
# ============================================================
# 步骤10（终极修复v3）：LTA 完整正确训练
#
# 上一版新报错原因：用"闭包函数"替换 target_layer.forward，闭包函数
# 是定义在另一个函数内部的局部对象，Python的pickle机制根本无法
# 序列化局部对象。Ultralytics每个epoch结束都会torch.save()整个
# 模型，一保存就报 AttributeError: Can't get local object。
#
# 修复方案：彻底放弃"打补丁式"的forward替换，改用正规的nn.Module
# 子类(LTAWrapper，定义在脚本顶层，不嵌套在任何函数内部)，把原始
# 层包裹在里面，再把backbone这个nn.Sequential里对应位置的层"整体
# 替换"成这个新模块实例。因为LTAWrapper是顶层类，pickle能正常
# 通过其模块路径找到它，可以被torch.save()正常序列化。
# ============================================================

import os
if not os.path.exists("/content/dataset"):
    raise SystemExit("检测到 /content/dataset 不存在，请先重跑步骤1、2，再重跑步骤6的清理部分")
DATA_ROOT = "/content/dataset"

import torch
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
        nn.init.zeros_(self.fc[2].weight)

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
        nn.init.zeros_(self.conv.weight)

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(out))


class LTA(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x


class LTAWrapper(nn.Module):
    """
    正规的nn.Module子类（顶层定义，可被pickle序列化）。
    包裹原始层 + LTA，替换掉backbone里的整个层对象，
    而不是打补丁式地替换forward方法。
    """
    def __init__(self, orig_layer, channels):
        super().__init__()
        self.orig_layer = orig_layer  # 原始层的参数(已加载好的预训练权重)被完整保留
        self.lta = LTA(channels=channels)
        # 复制Ultralytics内部路由需要的属性(.f表示输入来源，.i表示层索引)
        self.f = getattr(orig_layer, "f", -1)
        self.i = getattr(orig_layer, "i", None)
        self.type = getattr(orig_layer, "type", None)

    def forward(self, x):
        return self.lta(self.orig_layer(x))


def attach_lta(target_model, layer_idx, out_channels, device):
    backbone = target_model.model
    original_layer = backbone[layer_idx]
    wrapped = LTAWrapper(original_layer, out_channels).to(device)
    backbone[layer_idx] = wrapped  # 整体替换，不是打补丁
    return wrapped.lta


from ultralytics import YOLO

if not os.path.exists("yolov12n-cls.pt"):
    os.system('wget -q "https://github.com/sunsmarterjie/yolov12/releases/download/cls/yolov12n-cls.pt" -O yolov12n-cls.pt')

model_lta = YOLO("yolov12n-cls.pt")

_lta_status = {"injected": False, "in_optimizer": False, "ema_synced": False, "n_params_before": None, "n_params_after": None}

def inject_lta_callback(trainer):
    real_model = trainer.model
    backbone = real_model.model
    last_feature_idx = len(backbone) - 2
    target_layer = backbone[last_feature_idx]

    captured_shape = {}
    def probe_hook(module, inp, out):
        captured_shape["channels"] = out.shape[1]
    probe_handle = target_layer.register_forward_hook(probe_hook)

    device = next(real_model.parameters()).device
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    was_training = real_model.training
    real_model.eval()
    with torch.no_grad():
        real_model(dummy_input)
    real_model.train(was_training)
    probe_handle.remove()

    out_channels = captured_shape.get("channels")
    if out_channels is None:
        raise RuntimeError("动态探测失败，未能捕获输出通道数")

    lta_main = attach_lta(real_model, last_feature_idx, out_channels, device)
    _lta_status["injected"] = True
    print(f"[1/3] LTA已通过整体替换层对象的方式插入，第{last_feature_idx}层，通道数={out_channels}")

    if hasattr(trainer, "optimizer") and trainer.optimizer is not None:
        opt = trainer.optimizer
        _lta_status["n_params_before"] = sum(len(g["params"]) for g in opt.param_groups)
        target_group = None
        for g in opt.param_groups:
            if g.get("weight_decay", 0) and g["weight_decay"] > 0:
                target_group = g
                break
        if target_group is None:
            target_group = opt.param_groups[0]
        lta_params = list(lta_main.parameters())
        target_group["params"].extend(lta_params)
        _lta_status["n_params_after"] = sum(len(g["params"]) for g in opt.param_groups)
        _lta_status["in_optimizer"] = True
        print(f"[2/3] LTA的{len(lta_params)}个参数张量已追加进优化器现有参数组")
        print(f"      优化器追踪的参数张量总数: {_lta_status['n_params_before']} → {_lta_status['n_params_after']}")
    else:
        print("[2/3] ⚠️ 警告：未找到trainer.optimizer")

    if hasattr(trainer, "ema") and trainer.ema is not None and hasattr(trainer.ema, "ema"):
        attach_lta(trainer.ema.ema, last_feature_idx, out_channels, device)
        _lta_status["ema_synced"] = True
        print(f"[3/3] EMA副本已同步注入LTA结构（整体替换方式）")
    else:
        print("[3/3] ⚠️ 警告：未找到trainer.ema.ema")

    total_params = sum(p.numel() for p in real_model.parameters())
    lta_params_n = sum(p.numel() for p in lta_main.parameters())
    print(f"\nLTA模块自身参数量: {lta_params_n:,}")
    print(f"真实训练模型总参数量: {total_params:,}\n")

model_lta.add_callback("on_pretrain_routine_end", inject_lta_callback)

results_lta = model_lta.train(
    data=DATA_ROOT,
    epochs=50,
    imgsz=224,
    batch=32,
    patience=15,
    project="/content/drive/MyDrive/TCM_YOLOv12_runs",
    name="v12n_LTA_final_v3"
)

print("\n===== LTA注入状态自检 =====")
print(f"LTA已插入: {_lta_status['injected']}")
print(f"LTA已加入优化器: {_lta_status['in_optimizer']}")
print(f"追踪参数张量数变化: {_lta_status['n_params_before']} → {_lta_status['n_params_after']}")
print(f"EMA已同步: {_lta_status['ema_synced']}")

if _lta_status["injected"] and _lta_status["in_optimizer"] and _lta_status["ema_synced"]:
    print("\n✅ 全部检查通过，这次的结果才是真正有效的LTA消融实验")
else:
    print("\n⚠️ 有检查项未通过，请把完整日志发给我")

print("\n结果保存在 Drive 的 TCM_YOLOv12_runs/v12n_LTA_final_v3 文件夹")
print("请把最终 val Top-1 / Top-5 及'===== LTA注入状态自检 ====='部分发给我")

New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50     0.646G      3.614         29        224: 100% ━━━━━━━━━━━━ 62/62 2.2it/s 28.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 2.9it/s 3.4s
                   all      0.191      0.453

      Epoch    GPU_mem       loss  Instances       Size
       2/50     0.646G      3.431         32        224: 0% ──────────── 0/62  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50     0.648G      3.035         29        224: 100% ━━━━━━━━━━━━ 62/62 2.5it/s 24.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 2.3it/s 4.3s
                   all      0.311      0.675

      Epoch    GPU_mem       loss  Instances       Size
       3/50      0.65G      2.996         32        224: 0% ──────────── 0/62  0.3s

...

      46/50      0.65G       0.29         29        224: 100% ━━━━━━━━━━━━ 62/62 2.5it/s 24.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 2.8it/s 3.5s
                   all       0.76      0.931

      Epoch    GPU_mem       loss  Instances       Size
      47/50      0.65G     0.4196         32        224: 0% ──────────── 0/62  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      47/50      0.65G     0.2804         29        224: 100% ━━━━━━━━━━━━ 62/62 2.6it/s 24.2s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 2.2it/s 4.6s
                   all      0.751      0.938

      Epoch    GPU_mem       loss  Instances       Size
      48/50      0.65G     0.1847         32        224: 0% ──────────── 0/62  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      48/50      0.65G     0.3126         29        224: 100% ━━━━━━━━━━━━ 62/62 2.5it/s 25.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 3.6it/s 2.8s
                   all      0.759       0.94
EarlyStopping: Training stopped early as no improvement observed in last 15 epochs. Best results observed at epoch 33, best model saved as best.pt.
To update EarlyStopping(patience=15) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

48 epochs completed in 0.403 hours.
Optimizer stripped from /content/drive/MyDrive/TCM_YOLOv12_runs/v12n_LTA_final_v3/weights/last.pt, 3.7MB
Optimizer stripped from /content/drive/MyDrive/TCM_YOLOv12_runs/v12n_LTA_final_v3/weights/best.pt, 3.7MB

Validating /content/drive/MyDrive/TCM_YOLOv12_runs/v12n_LTA_final_v3/weights/best.pt...
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv12n-cls summary (fused): 102 layers, 1,737,937 parameters, 0 gradients, 6.

In [ ]:
# ============================================================
# 步骤11：YOLOv12s-cls + LTA 消融训练
#
# 复用YOLOv12n+LTA(final_v3)已验证有效的三处关键修复：
# 1. 动态探测真实输出通道数（不猜测层内部结构）
# 2. 用正规nn.Module整体替换层对象（支持pickle序列化，
#    避免闭包函数导致checkpoint保存失败）
# 3. 把LTA参数追加进优化器已有参数组（不新增组，避免
#    scheduler数量不匹配报错），并同步EMA副本结构
# ============================================================

import os
if not os.path.exists("/content/dataset"):
    raise SystemExit("检测到 /content/dataset 不存在，请先重跑步骤1、2，再重跑步骤6的清理部分")
DATA_ROOT = "/content/dataset"

import torch
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
        nn.init.zeros_(self.fc[2].weight)

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
        nn.init.zeros_(self.conv.weight)

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(out))


class LTA(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x


class LTAWrapper(nn.Module):
    def __init__(self, orig_layer, channels):
        super().__init__()
        self.orig_layer = orig_layer
        self.lta = LTA(channels=channels)
        self.f = getattr(orig_layer, "f", -1)
        self.i = getattr(orig_layer, "i", None)
        self.type = getattr(orig_layer, "type", None)

    def forward(self, x):
        return self.lta(self.orig_layer(x))


def attach_lta(target_model, layer_idx, out_channels, device):
    backbone = target_model.model
    original_layer = backbone[layer_idx]
    wrapped = LTAWrapper(original_layer, out_channels).to(device)
    backbone[layer_idx] = wrapped
    return wrapped.lta


from ultralytics import YOLO

if not os.path.exists("yolov12s-cls.pt"):
    os.system('wget -q "https://github.com/sunsmarterjie/yolov12/releases/download/cls/yolov12s-cls.pt" -O yolov12s-cls.pt')

if not os.path.exists("yolov12s-cls.pt") or os.path.getsize("yolov12s-cls.pt") < 1000000:
    raise SystemExit("yolov12s-cls.pt 下载失败或文件不完整，请检查网络连接")

model_lta_s = YOLO("yolov12s-cls.pt")

_lta_status = {"injected": False, "in_optimizer": False, "ema_synced": False, "n_params_before": None, "n_params_after": None}

def inject_lta_callback(trainer):
    real_model = trainer.model
    backbone = real_model.model
    last_feature_idx = len(backbone) - 2
    target_layer = backbone[last_feature_idx]

    captured_shape = {}
    def probe_hook(module, inp, out):
        captured_shape["channels"] = out.shape[1]
    probe_handle = target_layer.register_forward_hook(probe_hook)

    device = next(real_model.parameters()).device
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    was_training = real_model.training
    real_model.eval()
    with torch.no_grad():
        real_model(dummy_input)
    real_model.train(was_training)
    probe_handle.remove()

    out_channels = captured_shape.get("channels")
    if out_channels is None:
        raise RuntimeError("动态探测失败，未能捕获输出通道数")

    lta_main = attach_lta(real_model, last_feature_idx, out_channels, device)
    _lta_status["injected"] = True
    print(f"[1/3] LTA已通过整体替换层对象的方式插入，第{last_feature_idx}层，通道数={out_channels}")

    if hasattr(trainer, "optimizer") and trainer.optimizer is not None:
        opt = trainer.optimizer
        _lta_status["n_params_before"] = sum(len(g["params"]) for g in opt.param_groups)
        target_group = None
        for g in opt.param_groups:
            if g.get("weight_decay", 0) and g["weight_decay"] > 0:
                target_group = g
                break
        if target_group is None:
            target_group = opt.param_groups[0]
        lta_params = list(lta_main.parameters())
        target_group["params"].extend(lta_params)
        _lta_status["n_params_after"] = sum(len(g["params"]) for g in opt.param_groups)
        _lta_status["in_optimizer"] = True
        print(f"[2/3] LTA的{len(lta_params)}个参数张量已追加进优化器现有参数组")
        print(f"      优化器追踪的参数张量总数: {_lta_status['n_params_before']} → {_lta_status['n_params_after']}")
    else:
        print("[2/3] ⚠️ 警告：未找到trainer.optimizer")

    if hasattr(trainer, "ema") and trainer.ema is not None and hasattr(trainer.ema, "ema"):
        attach_lta(trainer.ema.ema, last_feature_idx, out_channels, device)
        _lta_status["ema_synced"] = True
        print(f"[3/3] EMA副本已同步注入LTA结构")
    else:
        print("[3/3] ⚠️ 警告：未找到trainer.ema.ema")

    total_params = sum(p.numel() for p in real_model.parameters())
    lta_params_n = sum(p.numel() for p in lta_main.parameters())
    print(f"\nLTA模块自身参数量: {lta_params_n:,}")
    print(f"真实训练模型总参数量: {total_params:,}\n")

model_lta_s.add_callback("on_pretrain_routine_end", inject_lta_callback)

results_lta_s = model_lta_s.train(
    data=DATA_ROOT,
    epochs=50,
    imgsz=224,
    batch=32,
    patience=15,
    project="/content/drive/MyDrive/TCM_YOLOv12_runs",
    name="v12s_LTA_final"
)

print("\n===== LTA注入状态自检 =====")
print(f"LTA已插入: {_lta_status['injected']}")
print(f"LTA已加入优化器: {_lta_status['in_optimizer']}")
print(f"追踪参数张量数变化: {_lta_status['n_params_before']} → {_lta_status['n_params_after']}")
print(f"EMA已同步: {_lta_status['ema_synced']}")

if _lta_status["injected"] and _lta_status["in_optimizer"] and _lta_status["ema_synced"]:
    print("\n✅ 全部检查通过，这次的结果才是真正有效的LTA消融实验")
else:
    print("\n⚠️ 有检查项未通过，请把完整日志发给我")

print("\n结果保存在 Drive 的 TCM_YOLOv12_runs/v12s_LTA_final 文件夹")
print("请把最终 val Top-1 / Top-5 及'===== LTA注入状态自检 ====='部分发给我")
print("与标准 YOLOv12s-cls 基线 (80.8%/95.6%) 对比")

New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      1.21G       3.56         29        224: 100% ━━━━━━━━━━━━ 62/62 2.0it/s 31.1s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 3.5it/s 2.8s
                   all      0.229      0.502

      Epoch    GPU_mem       loss  Instances       Size
       2/50      1.26G      3.271         32        224: 0% ──────────── 0/62  0.4s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      1.27G      2.719         29        224: 100% ━━━━━━━━━━━━ 62/62 2.3it/s 26.6s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 4.2it/s 2.4s
                   all      0.401      0.757

      Epoch    GPU_mem       loss  Instances       Size
       3/50      1.27G      2.114         32        224: 0% ──────────── 0/62  0.4s

...

      48/50      1.27G    0.09875         29        224: 100% ━━━━━━━━━━━━ 62/62 2.3it/s 27.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 5.6it/s 1.8s
                   all      0.798      0.953

      Epoch    GPU_mem       loss  Instances       Size
      49/50      1.27G    0.02535         32        224: 0% ──────────── 0/62  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      49/50      1.27G    0.08094         29        224: 100% ━━━━━━━━━━━━ 62/62 2.2it/s 27.8s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 3.9it/s 2.6s
                   all      0.789      0.957

      Epoch    GPU_mem       loss  Instances       Size
      50/50      1.27G     0.1014         32        224: 0% ──────────── 0/62  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      50/50      1.27G    0.08766         29        224: 100% ━━━━━━━━━━━━ 62/62 2.3it/s 27.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 3.5it/s 2.8s
                   all      0.798      0.957

50 epochs completed in 0.450 hours.
Optimizer stripped from /content/drive/MyDrive/TCM_YOLOv12_runs/v12s_LTA_final/weights/last.pt, 12.4MB
Optimizer stripped from /content/drive/MyDrive/TCM_YOLOv12_runs/v12s_LTA_final/weights/best.pt, 12.4MB

Validating /content/drive/MyDrive/TCM_YOLOv12_runs/v12s_LTA_final/weights/best.pt...
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv12s-cls summary (fused): 102 layers, 6,051,097 parameters, 0 gradients, 17.3 GFLOPs
train: /content/dataset/train... found 1981 images in 39 classes ✅ 
val: /content/dataset/val... found 634 images in 39 classes ✅ 
test: /content/dataset/test... found 636 images in 39 classes ✅ 
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 3.0it/s 3

In [ ]:
# ============================================================
# 步骤12：在 test 集上对全部5个模型做最终评估
#
# 之前所有Top-1/Top-5都是在val集上得到的(训练时用来调参和早停
# 判断)。论文Results表应该报告test集上的最终指标，因为test集
# 从未参与任何训练决策，结果更有说服力。
#
# 注意：加了LTA的两个模型的best.pt里包含自定义的LTAWrapper结构，
# 必须先在当前会话里重新定义这些类，PyTorch才能正确反序列化
# checkpoint，否则会报"can't find class"之类的错误。
# ============================================================

import os
if not os.path.exists("/content/dataset"):
    raise SystemExit("检测到 /content/dataset 不存在，请先重跑步骤1、2，再重跑步骤6的清理部分")
DATA_ROOT = "/content/dataset"

import torch
import torch.nn as nn

# ===== 必须重新定义，用于正确加载LTA模型的checkpoint =====
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(out))


class LTA(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x


class LTAWrapper(nn.Module):
    def __init__(self, orig_layer, channels):
        super().__init__()
        self.orig_layer = orig_layer
        self.lta = LTA(channels=channels)
        self.f = getattr(orig_layer, "f", -1)
        self.i = getattr(orig_layer, "i", None)
        self.type = getattr(orig_layer, "type", None)

    def forward(self, x):
        return self.lta(self.orig_layer(x))


from ultralytics import YOLO
import pandas as pd

MODELS = {
    "YOLOv11n-cls":        "/content/drive/MyDrive/TCM_YOLOv11_runs/full_train_baseline/weights/best.pt",
    "YOLOv12n-cls":        "/content/drive/MyDrive/TCM_YOLOv12_runs/v12n_baseline/weights/best.pt",
    "YOLOv12n-cls+LTA":    "/content/drive/MyDrive/TCM_YOLOv12_runs/v12n_LTA_final_v3/weights/best.pt",
    "YOLOv12s-cls":        "/content/drive/MyDrive/TCM_YOLOv12_runs/v12s_baseline/weights/best.pt",
    "YOLOv12s-cls+LTA":    "/content/drive/MyDrive/TCM_YOLOv12_runs/v12s_LTA_final/weights/best.pt",
}

results_table = []

for name, weight_path in MODELS.items():
    print(f"\n{'='*60}")
    print(f"评估: {name}")
    print(f"{'='*60}")

    if not os.path.exists(weight_path):
        print(f"⚠️ 权重文件不存在: {weight_path}，跳过")
        results_table.append({
            "model": name, "test_top1": None, "test_top5": None,
            "params": None, "status": "权重文件缺失"
        })
        continue

    try:
        model = YOLO(weight_path)
        metrics = model.val(data=DATA_ROOT, split="test", imgsz=224, batch=32)

        top1 = float(metrics.top1)
        top5 = float(metrics.top5)
        n_params = sum(p.numel() for p in model.model.parameters())

        print(f"\n{name}: Test Top-1={top1:.4f}  Test Top-5={top5:.4f}  参数量={n_params:,}")

        results_table.append({
            "model": name,
            "test_top1": round(top1 * 100, 2),
            "test_top5": round(top5 * 100, 2),
            "params": n_params,
            "status": "成功"
        })
    except Exception as e:
        print(f"⚠️ 评估失败: {e}")
        results_table.append({
            "model": name, "test_top1": None, "test_top5": None,
            "params": None, "status": f"失败: {str(e)[:100]}"
        })

df = pd.DataFrame(results_table)
print("\n\n" + "="*60)
print("测试集终评汇总表")
print("="*60)
print(df.to_string(index=False))

csv_path = "/content/drive/MyDrive/TCM_YOLOv12_runs/test_set_final_results.csv"
df.to_csv(csv_path, index=False)
print(f"\n结果已保存到: {csv_path}")
print("请把上面的汇总表完整发给我")


评估: YOLOv11n-cls
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-cls summary (fused): 47 layers, 1,575,983 parameters, 0 gradients, 3.2 GFLOPs
train: /content/dataset/train... found 1981 images in 39 classes ✅ 
val: /content/dataset/val... found 634 images in 39 classes ✅ 
test: /content/dataset/test... found 636 images in 39 classes ✅ 
WARNING ⚠️ test: Slow image access detected (ping: 0.0±0.0 ms, read: 14.7±7.2 MB/s, size: 15.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
test: Scanning /content/dataset/test... 636 images, 0 corrupt: 100% ━━━━━━━━━━━━ 636/636 977.2it/s 0.7s
test: /content/dataset/test/baiguo/9100468eeb239f16e6ea8dff496a40bf.jpg: corrupt JPEG restored and saved
test: /content/dataset/test/beidougen/b0b27ae3c898bcbf3d2653cd5acd527d.jpg: corrupt JPEG restored and saved
test: /content/dataset/test/caowu/7d04afe860925777d149469817cc

In [ ]:
# ============================================================
# 步骤13+14 合并版：先统计LTA真实纠错效果，再对"真正被救回"的
# 案例做Grad-CAM可视化。彻底替代原步骤13（已废弃：原方法只挑了
# 标准模型判错的图，从未验证LTA是否真的判对，导致视觉证据和
# 因果关系脱节）。
#
# 流程：
# 1. 对test集全部图片，同时记录标准模型和LTA模型的预测结果
# 2. 统计四类情况：都对/都错/被LTA救回(标准错→LTA对)/被LTA新
#    引入错误(标准对→LTA错)
# 3. 只对"被LTA救回"的真实案例生成三联图：原图 | 标准模型注意力
#    (目标设为真实类别，展示它为何没能聚焦对) | LTA模型注意力
#    (目标同样设为真实类别，展示它聚焦到了什么，才判断正确)
# ============================================================

import subprocess
subprocess.run(["pip", "install", "-q", "grad-cam"], check=True)

import os
if not os.path.exists("/content/dataset"):
    raise SystemExit("检测到 /content/dataset 不存在，请先重跑步骤1、2，再重跑步骤6的清理部分")
DATA_ROOT = "/content/dataset"

import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(out))


class LTA(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x


class LTAWrapper(nn.Module):
    def __init__(self, orig_layer, channels):
        super().__init__()
        self.orig_layer = orig_layer
        self.lta = LTA(channels=channels)
        self.f = getattr(orig_layer, "f", -1)
        self.i = getattr(orig_layer, "i", None)
        self.type = getattr(orig_layer, "type", None)

    def forward(self, x):
        return self.lta(self.orig_layer(x))


class LogitsOnlyWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        out = self.base_model(x)
        if isinstance(out, (tuple, list)):
            return out[-1]
        return out


from ultralytics import YOLO
from ultralytics.data.augment import classify_transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

STD_WEIGHT = "/content/drive/MyDrive/TCM_YOLOv12_runs/v12s_baseline/weights/best.pt"
LTA_WEIGHT = "/content/drive/MyDrive/TCM_YOLOv12_runs/v12s_LTA_final/weights/best.pt"

model_std = YOLO(STD_WEIGHT)
model_lta = YOLO(LTA_WEIGHT)
class_names = model_std.names
name_to_idx = {v: k for k, v in class_names.items()}

torch_std_raw = model_std.model.to(device).eval()
torch_lta_raw = model_lta.model.to(device).eval()

torch_std = LogitsOnlyWrapper(torch_std_raw).to(device).eval()
torch_lta = LogitsOnlyWrapper(torch_lta_raw).to(device).eval()

transform = classify_transforms(224)

# ===== 第一步：全量对比统计 =====
print("正在对test集全部图片，同时用标准模型和LTA模型做预测对比...\n")
test_dir = os.path.join(DATA_ROOT, "test")
results = []

with torch.no_grad():
    for cls_name in sorted(os.listdir(test_dir)):
        cls_dir = os.path.join(test_dir, cls_name)
        if not os.path.isdir(cls_dir):
            continue
        true_idx = name_to_idx.get(cls_name)
        if true_idx is None:
            continue
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            img_path = os.path.join(cls_dir, fname)
            try:
                img = Image.open(img_path).convert("RGB")
                tensor = transform(img).unsqueeze(0).to(device)
                std_pred = int(torch_std(tensor).argmax(dim=1))
                lta_pred = int(torch_lta(tensor).argmax(dim=1))
                results.append((img_path, true_idx, std_pred, lta_pred))
            except Exception:
                continue

n_total = len(results)
std_correct = sum(1 for _, t, s, l in results if s == t)
lta_correct = sum(1 for _, t, s, l in results if l == t)
rescued = [(p, t, s, l) for p, t, s, l in results if s != t and l == t]
newly_broken = [(p, t, s, l) for p, t, s, l in results if s == t and l != t]
both_wrong = [(p, t, s, l) for p, t, s, l in results if s != t and l != t]
both_right = sum(1 for _, t, s, l in results if s == t and l == t)

print(f"===== 全量对比统计 =====")
print(f"测试集总数: {n_total}")
print(f"标准模型正确数: {std_correct} ({std_correct/n_total*100:.2f}%)")
print(f"LTA模型正确数:   {lta_correct} ({lta_correct/n_total*100:.2f}%)")
print(f"两模型都判对: {both_right}")
print(f"标准错→LTA对（被LTA救回）: {len(rescued)}")
print(f"标准对→LTA错（被LTA新引入的错误）: {len(newly_broken)}")
print(f"两模型都判错: {len(both_wrong)}")
print(f"净纠错效果: +{len(rescued)} 救回, -{len(newly_broken)} 新错 = 净改善 {len(rescued)-len(newly_broken)} 张")

if not rescued:
    print("\n⚠️ 没有找到任何LTA成功救回的案例，无法生成有因果说服力的Grad-CAM对比图")
    print("建议改用'两模型都判错但LTA置信度更接近正确答案'这类次优证据，请把上面统计结果发给我讨论下一步")
    raise SystemExit()

print(f"\n===== LTA成功救回的具体案例（标准错→LTA对，共{len(rescued)}个）=====")
for p, t, s, l in rescued:
    print(f"  {os.path.basename(p)}: 真实={class_names[t]}, 标准误判为={class_names[s]}, LTA判对")

# ===== 第二步：只对"被救回"的案例生成Grad-CAM三联图 =====
target_layer_std = [torch_std_raw.model[-2]]
target_layer_lta = [torch_lta_raw.model[-2]]
cam_std = GradCAM(model=torch_std, target_layers=target_layer_std)
cam_lta = GradCAM(model=torch_lta, target_layers=target_layer_lta)

output_dir = "/content/drive/MyDrive/TCM_YOLOv12_runs/gradcam_rescued_cases"
os.makedirs(output_dir, exist_ok=True)

saved_count = 0
for img_path, true_idx, std_pred, lta_pred in rescued[:10]:  # 最多展示10个
    img = Image.open(img_path).convert("RGB")
    img_resized = img.resize((224, 224))
    rgb_float = np.array(img_resized).astype(np.float32) / 255.0

    target = [ClassifierOutputTarget(true_idx)]

    tensor_std = transform(img).unsqueeze(0).to(device)
    tensor_std.requires_grad_(True)
    grayscale_cam_std = cam_std(input_tensor=tensor_std, targets=target)[0]

    tensor_lta = transform(img).unsqueeze(0).to(device)
    tensor_lta.requires_grad_(True)
    grayscale_cam_lta = cam_lta(input_tensor=tensor_lta, targets=target)[0]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(rgb_float)
    axes[0].set_title(f"Original\nTrue: {class_names[true_idx]}\nStandard misclassified as: {class_names[std_pred]}\n(LTA correctly classified)")
    axes[0].axis("off")

    axes[1].imshow(rgb_float)
    axes[1].imshow(grayscale_cam_std, cmap="jet", alpha=0.5)
    axes[1].set_title("YOLOv12s-cls (Standard) - Wrong")
    axes[1].axis("off")

    axes[2].imshow(rgb_float)
    axes[2].imshow(grayscale_cam_lta, cmap="jet", alpha=0.5)
    axes[2].set_title("YOLOv12s-cls + LTA - Correct")
    axes[2].axis("off")

    plt.tight_layout()
    save_name = f"rescued_{class_names[true_idx]}_{saved_count}.png"
    save_path = os.path.join(output_dir, save_name)
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    saved_count += 1
    print(f"已保存: {save_name}")

print(f"\n共生成 {saved_count} 组'LTA真实救回案例'对比图，保存在: {output_dir}")
print("这些才是有因果说服力的样本：标准模型看错了地方判错，LTA看对了地方判对")

正在对test集全部图片，同时用标准模型和LTA模型做预测对比...

===== 全量对比统计 =====
测试集总数: 636
标准模型正确数: 524 (82.39%)
LTA模型正确数:   533 (83.81%)
两模型都判对: 509
标准错→LTA对（被LTA救回）: 24
标准对→LTA错（被LTA新引入的错误）: 15
两模型都判错: 88
净纠错效果: +24 救回, -15 新错 = 净改善 9 张

===== LTA成功救回的具体案例（标准错→LTA对，共24个）=====
  16.jpg: 真实=baifuzi, 标准误判为=gansui, LTA判对
  image19.jpeg: 真实=baiguo, 标准误判为=changshan, LTA判对
  34e0001cbaf9c083a29f6a3a0541817b.jpeg: 真实=banxia, 标准误判为=tiannanxing, LTA判对
  fd6b2bb6a6fd68ab97d2d8c698311975.jpg: 真实=banxia, 标准误判为=tiannanxing, LTA判对
  4.jpg: 真实=banxia, 标准误判为=qianniuzi, LTA判对
  5753008f15efb1335370d12c8401a979.jpg: 真实=banxia, 标准误判为=tiannanxing, LTA判对
  38d37a216173b72d37ee8fedb8b10d4e.jpeg: 真实=banxia, 标准误判为=tiannanxing, LTA判对
  9bd5eb8c33739a0ae5e33667a874b85b.jpg: 真实=beidougen, 标准误判为=liangmianzhen, LTA判对
  591d1c41484927693a68e9306cf112f9.jpg: 真实=caowu, 标准误判为=xianmao, LTA判对
  6df7fc146123b9943a5e6c5f529bd7a4.jpeg: 真实=caowu, 标准误判为=hongdaji, LTA判对
  2711996a45155033ea5a8ba57017945f.jpeg: 真实=chonglou, 标准误判为=heshi, LTA判对
  3dc41

In [ ]:
# ============================================================
# 步骤15：数据增强强度对比实验（无增强 / 轻度增强 / 强增强）
#
# 目的：验证原论文发现"强增强反而损害细粒度纹理判别"这一结论
# 在39类新数据上是否依然成立。用YOLOv12s-cls作为载体(目前五组
# 实验里表现最好、最有代表性的模型)。
#
# 三组设置：
# - 无增强: 仅resize+normalize，关闭所有几何/色彩扰动
# - 轻度增强: 仅水平翻转(fliplr=0.5)，其余关闭
# - 强增强: 随机裁剪(scale) + 旋转(degrees) + 色彩抖动(hsv) + mosaic
# ============================================================

import os
if not os.path.exists("/content/dataset"):
    raise SystemExit("检测到 /content/dataset 不存在，请先重跑步骤1、2，再重跑步骤6的清理部分")
DATA_ROOT = "/content/dataset"

if not os.path.exists("yolov12s-cls.pt"):
    os.system('wget -q "https://github.com/sunsmarterjie/yolov12/releases/download/cls/yolov12s-cls.pt" -O yolov12s-cls.pt')

if not os.path.exists("yolov12s-cls.pt") or os.path.getsize("yolov12s-cls.pt") < 1000000:
    raise SystemExit("yolov12s-cls.pt 下载失败或文件不完整，请检查网络连接")

from ultralytics import YOLO

AUGMENTATION_CONFIGS = {
    "no_aug": {
        "hsv_h": 0.0, "hsv_s": 0.0, "hsv_v": 0.0,
        "degrees": 0.0, "translate": 0.0, "scale": 0.0, "shear": 0.0,
        "perspective": 0.0, "flipud": 0.0, "fliplr": 0.0,
        "mosaic": 0.0, "mixup": 0.0, "cutmix": 0.0, "copy_paste": 0.0,
        "erasing": 0.0, "auto_augment": None,
    },
    "light_aug": {
        "hsv_h": 0.0, "hsv_s": 0.0, "hsv_v": 0.0,
        "degrees": 0.0, "translate": 0.0, "scale": 0.0, "shear": 0.0,
        "perspective": 0.0, "flipud": 0.0, "fliplr": 0.5,
        "mosaic": 0.0, "mixup": 0.0, "cutmix": 0.0, "copy_paste": 0.0,
        "erasing": 0.0, "auto_augment": None,
    },
    "strong_aug": {
        "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
        "degrees": 15.0, "translate": 0.1, "scale": 0.5, "shear": 5.0,
        "perspective": 0.0005, "flipud": 0.0, "fliplr": 0.5,
        "mosaic": 1.0, "mixup": 0.1, "cutmix": 0.1, "copy_paste": 0.0,
        "erasing": 0.4, "auto_augment": "randaugment",
    },
}

results_summary = []

for aug_name, aug_params in AUGMENTATION_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"训练配置: {aug_name}")
    print(f"{'='*60}")

    model = YOLO("yolov12s-cls.pt")
    results = model.train(
        data=DATA_ROOT,
        epochs=50,
        imgsz=224,
        batch=32,
        patience=15,
        project="/content/drive/MyDrive/TCM_Augmentation_Study",
        name=aug_name,
        **aug_params,
    )

    metrics = model.val(data=DATA_ROOT, split="test", imgsz=224, batch=32)
    test_top1 = float(metrics.top1) * 100
    test_top5 = float(metrics.top5) * 100

    print(f"\n{aug_name} 完成: Test Top-1={test_top1:.2f}%  Test Top-5={test_top5:.2f}%")
    results_summary.append({
        "augmentation": aug_name,
        "test_top1": round(test_top1, 2),
        "test_top5": round(test_top5, 2),
    })

import pandas as pd
df = pd.DataFrame(results_summary)
print("\n\n" + "="*60)
print("数据增强强度对比汇总表")
print("="*60)
print(df.to_string(index=False))

csv_path = "/content/drive/MyDrive/TCM_Augmentation_Study/augmentation_comparison_results.csv"
df.to_csv(csv_path, index=False)
print(f"\n结果已保存到: {csv_path}")
print("请把上面的汇总表完整发给我")


训练配置: no_aug
New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

TypeError: ERROR ❌️ /content/drive/MyDrive/TCM_Augmentation_Study/strong_aug/weights/best.pt is not a loadable checkpoint — the file is empty, truncated or corrupted (RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory. This is an internal miniz error. If you are seeing this error, there is a high likelihood that your checkpoint file is corrupted. This can happen if the checkpoint was not saved properly, was transferred incorrectly, or the file was modified after saving.).
Recommend fixes are to re-download or re-export the file, or to run a command with an official Ultralytics model, i.e. 'yolo predict model=yolo26n.pt'

In [ ]:
# ============================================================
# 步骤15c：重跑strong_aug，先存本地再复制到Drive，避免重蹈覆辙
#
# 上次问题根源：Ultralytics训练结束后会重新加载best.pt、剥离
# 优化器状态、重写回磁盘("stripped"步骤)，这次重写操作直接对着
# Google Drive挂载路径写，撞上了Drive同步的竞态条件导致文件损坏。
#
# 修复策略：先让Ultralytics把结果保存在Colab本地磁盘(/content/)，
# 训练完全结束、文件确认完整后，再用shutil.copy手动复制到Drive，
# 这样"stripped重写"这个关键步骤完全在本地磁盘完成，不会被Drive
# 同步延迟干扰。
#
# 这样重跑后能拿到strong_aug真正的best.pt，与no_aug/light_aug
# 用相同标准(best.pt)公平对比。
# ============================================================

import os
import shutil
if not os.path.exists("/content/dataset"):
    raise SystemExit("检测到 /content/dataset 不存在，请先重跑步骤1、2，再重跑步骤6的清理部分")
DATA_ROOT = "/content/dataset"

if not os.path.exists("yolov12s-cls.pt"):
    os.system('wget -q "https://github.com/sunsmarterjie/yolov12/releases/download/cls/yolov12s-cls.pt" -O yolov12s-cls.pt')

from ultralytics import YOLO

strong_aug_params = {
    "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
    "degrees": 15.0, "translate": 0.1, "scale": 0.5, "shear": 5.0,
    "perspective": 0.0005, "flipud": 0.0, "fliplr": 0.5,
    "mosaic": 1.0, "mixup": 0.1, "cutmix": 0.1, "copy_paste": 0.0,
    "erasing": 0.4, "auto_augment": "randaugment",
}

LOCAL_PROJECT = "/content/local_augmentation_runs"

model = YOLO("yolov12s-cls.pt")
results = model.train(
    data=DATA_ROOT,
    epochs=50,
    imgsz=224,
    batch=32,
    patience=15,
    project=LOCAL_PROJECT,   # 关键改动：先存本地，不直接写Drive
    name="strong_aug_v2",
    **strong_aug_params,
)

local_best = f"{LOCAL_PROJECT}/strong_aug_v2/weights/best.pt"
local_last = f"{LOCAL_PROJECT}/strong_aug_v2/weights/last.pt"

print(f"\n本地best.pt大小: {os.path.getsize(local_best)/1024/1024:.2f} MB")
print(f"本地last.pt大小: {os.path.getsize(local_last)/1024/1024:.2f} MB")

# 验证本地文件完整可读
import torch
try:
    torch.load(local_best, map_location="cpu", weights_only=False)
    print("✅ 本地best.pt验证完整可读")
except Exception as e:
    raise SystemExit(f"⚠️ 本地best.pt仍然损坏，问题不是Drive导致的: {e}")

# 确认完整后，再复制到Drive长期保存
DRIVE_DEST = "/content/drive/MyDrive/TCM_Augmentation_Study/strong_aug_v2"
os.makedirs(f"{DRIVE_DEST}/weights", exist_ok=True)
shutil.copy(local_best, f"{DRIVE_DEST}/weights/best.pt")
shutil.copy(local_last, f"{DRIVE_DEST}/weights/last.pt")
print(f"\n已复制到Drive: {DRIVE_DEST}/weights/")

# 用确认完整的本地best.pt做最终评估
metrics = model.val(data=DATA_ROOT, split="test", imgsz=224, batch=32)
test_top1 = float(metrics.top1) * 100
test_top5 = float(metrics.top5) * 100
print(f"\nstrong_aug(v2，使用真正的best.pt) 最终结果:")
print(f"Test Top-1={test_top1:.2f}%  Test Top-5={test_top5:.2f}%")

import pandas as pd
final_results = [
    {"augmentation": "no_aug", "test_top1": 77.20, "test_top5": 94.81, "checkpoint_used": "best.pt"},
    {"augmentation": "light_aug", "test_top1": 78.93, "test_top5": 94.97, "checkpoint_used": "best.pt"},
    {"augmentation": "strong_aug", "test_top1": round(test_top1, 2), "test_top5": round(test_top5, 2), "checkpoint_used": "best.pt(重跑，验证完整)"},
]
df = pd.DataFrame(final_results)
print("\n\n" + "="*70)
print("数据增强强度对比完整汇总表（三组均使用best.pt，公平对比）")
print("="*70)
print(df.to_string(index=False))

csv_path = "/content/drive/MyDrive/TCM_Augmentation_Study/augmentation_comparison_final_v2.csv"
df.to_csv(csv_path, index=False)
print(f"\n结果已保存到: {csv_path}")
print("请把上面的完整汇总表发给我")

New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.1, data=/content/dataset, degrees=15.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo

In [ ]:
# ============================================================
# 步骤16：全39类混淆矩阵
#
# 为最佳模型(YOLOv12s-cls+LTA)和标准基线(YOLOv12s-cls)分别生成
# 完整的39x39混淆矩阵，用于Results部分的可视化配图，并直接支撑
# 之前发现的"混淆集中在三生药群"这一结论。
# ============================================================

import os
if not os.path.exists("/content/dataset"):
    raise SystemExit("检测到 /content/dataset 不存在，请先重跑步骤1、2，再重跑步骤6的清理部分")
DATA_ROOT = "/content/dataset"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(out))


class LTA(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x


class LTAWrapper(nn.Module):
    def __init__(self, orig_layer, channels):
        super().__init__()
        self.orig_layer = orig_layer
        self.lta = LTA(channels=channels)
        self.f = getattr(orig_layer, "f", -1)
        self.i = getattr(orig_layer, "i", None)
        self.type = getattr(orig_layer, "type", None)

    def forward(self, x):
        return self.lta(self.orig_layer(x))


class LogitsOnlyWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        out = self.base_model(x)
        if isinstance(out, (tuple, list)):
            return out[-1]
        return out


from ultralytics import YOLO
from ultralytics.data.augment import classify_transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = classify_transforms(224)

MODELS_FOR_CM = {
    "YOLOv12s-cls (Standard)": "/content/drive/MyDrive/TCM_YOLOv12_runs/v12s_baseline/weights/best.pt",
    "YOLOv12s-cls + LTA": "/content/drive/MyDrive/TCM_YOLOv12_runs/v12s_LTA_final/weights/best.pt",
}

def get_predictions(weight_path):
    model = YOLO(weight_path)
    class_names = model.names
    name_to_idx = {v: k for k, v in class_names.items()}
    torch_model = LogitsOnlyWrapper(model.model.to(device).eval()).to(device).eval()

    test_dir = os.path.join(DATA_ROOT, "test")
    y_true, y_pred = [], []

    with torch.no_grad():
        for cls_name in sorted(os.listdir(test_dir)):
            cls_dir = os.path.join(test_dir, cls_name)
            if not os.path.isdir(cls_dir):
                continue
            true_idx = name_to_idx.get(cls_name)
            if true_idx is None:
                continue
            for fname in os.listdir(cls_dir):
                if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    continue
                img_path = os.path.join(cls_dir, fname)
                try:
                    img = Image.open(img_path).convert("RGB")
                    tensor = transform(img).unsqueeze(0).to(device)
                    pred_idx = int(torch_model(tensor).argmax(dim=1))
                    y_true.append(true_idx)
                    y_pred.append(pred_idx)
                except Exception:
                    continue

    return y_true, y_pred, class_names

def plot_confusion_matrix(y_true, y_pred, class_names, title, save_path):
    n_classes = len(class_names)
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1

    # 按行归一化(每个真实类别内部的预测分布比例)，更适合可视化不同样本量的类别
    cm_norm = cm.astype(float)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_norm = cm_norm / row_sums

    labels = [class_names[i] for i in range(n_classes)]

    fig, ax = plt.subplots(figsize=(18, 16))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(n_classes))
    ax.set_yticks(range(n_classes))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label="Proportion (row-normalized)")

    # 只标注非零且非对角线的误判格子，避免图面过于拥挤
    for i in range(n_classes):
        for j in range(n_classes):
            if i != j and cm[i, j] > 0:
                ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=6, color="red")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"已保存: {save_path}")
    return cm

output_dir = "/content/drive/MyDrive/TCM_YOLOv12_runs/confusion_matrices"
os.makedirs(output_dir, exist_ok=True)

all_cms = {}
for model_name, weight_path in MODELS_FOR_CM.items():
    print(f"\n{'='*60}")
    print(f"生成混淆矩阵: {model_name}")
    print(f"{'='*60}")

    if not os.path.exists(weight_path):
        print(f"⚠️ 权重文件不存在: {weight_path}，跳过")
        continue

    y_true, y_pred, class_names = get_predictions(weight_path)
    accuracy = sum(1 for t, p in zip(y_true, y_pred) if t == p) / len(y_true)
    print(f"验证: 共{len(y_true)}张图，Top-1={accuracy*100:.2f}%")

    safe_name = model_name.replace(" ", "_").replace("(", "").replace(")", "").replace("+", "plus")
    save_path = os.path.join(output_dir, f"confusion_matrix_{safe_name}.png")
    cm = plot_confusion_matrix(y_true, y_pred, class_names, f"Confusion Matrix - {model_name}", save_path)
    all_cms[model_name] = (cm, class_names)

# 找出全局最容易混淆的类别对(基于LTA模型的混淆矩阵，即最终模型)
if "YOLOv12s-cls + LTA" in all_cms:
    cm, class_names = all_cms["YOLOv12s-cls + LTA"]
    n_classes = len(class_names)
    pairs = []
    for i in range(n_classes):
        for j in range(n_classes):
            if i != j and cm[i, j] > 0:
                pairs.append((cm[i, j], class_names[i], class_names[j]))
    pairs.sort(reverse=True)

    print(f"\n===== 全局最容易混淆的类别对 Top-10 (YOLOv12s-cls+LTA, 全39类混淆矩阵) =====")
    for count, true_c, pred_c in pairs[:10]:
        print(f"  {true_c} → 误判为 {pred_c}: {count}次")

print(f"\n全部混淆矩阵已保存到: {output_dir}")
print("请把上面打印的验证准确率、以及Top-10混淆类别对列表发给我")


生成混淆矩阵: YOLOv12s-cls (Standard)
验证: 共636张图，Top-1=82.39%
已保存: /content/drive/MyDrive/TCM_YOLOv12_runs/confusion_matrices/confusion_matrix_YOLOv12s-cls_Standard.png

生成混淆矩阵: YOLOv12s-cls + LTA
验证: 共636张图，Top-1=83.81%
已保存: /content/drive/MyDrive/TCM_YOLOv12_runs/confusion_matrices/confusion_matrix_YOLOv12s-cls_plus_LTA.png

===== 全局最容易混淆的类别对 Top-10 (YOLOv12s-cls+LTA, 全39类混淆矩阵) =====
  tiannanxing → 误判为 banxia: 4次
  xiangjiapi → 误判为 tujingpi: 3次
  tujingpi → 误判为 xiangjiapi: 3次
  jiulixiang → 误判为 kulianpi: 3次
  yuanhua → 误判为 heshi: 2次
  tujingpi → 误判为 beidougen: 2次
  shancigu → 误判为 chuanlianzi: 2次
  naoyanghua → 误判为 yuanhua: 2次
  langdu → 误判为 shanglu: 2次
  huajiao → 误判为 wuzhuyu: 2次

全部混淆矩阵已保存到: /content/drive/MyDrive/TCM_YOLOv12_runs/confusion_matrices
请把上面打印的验证准确率、以及Top-10混淆类别对列表发给我


In [8]:
# ============================================================
# 步骤17：合成鲁棒性测试（替代已废弃的test_subset）
#
# 对现有test集(636张)施加三种模拟真实拍摄条件的合成扰动：
# 1. 高斯模糊 - 模拟手机拍摄失焦
# 2. 高斯噪声 - 模拟低照度下的传感器噪点
# 3. 亮度/对比度变化 - 模拟不同药房灯光环境
#
# 对比YOLOv12s-cls标准版 vs +LTA版，在每种扰动下的精度衰减幅度，
# 验证LTA是否在图像质量下降的场景下表现出更强的鲁棒性。
# 不需要重新训练，复用已有权重，仅需构造扰动图像+跑评估。
# ============================================================

import os
import shutil
if not os.path.exists("/content/dataset"):
    raise SystemExit("检测到 /content/dataset 不存在，请先重跑步骤1、2，再重跑步骤6的清理部分")
DATA_ROOT = "/content/dataset"

import torch
import torch.nn as nn
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(out))


class LTA(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x


class LTAWrapper(nn.Module):
    def __init__(self, orig_layer, channels):
        super().__init__()
        self.orig_layer = orig_layer
        self.lta = LTA(channels=channels)
        self.f = getattr(orig_layer, "f", -1)
        self.i = getattr(orig_layer, "i", None)
        self.type = getattr(orig_layer, "type", None)

    def forward(self, x):
        return self.lta(self.orig_layer(x))


class LogitsOnlyWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        out = self.base_model(x)
        if isinstance(out, (tuple, list)):
            return out[-1]
        return out


from ultralytics import YOLO
from ultralytics.data.augment import classify_transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = classify_transforms(224)

# ===== 第一步：构造三种扰动版本的test集(仅生成图像，不复制到磁盘，直接内存处理) =====

def apply_gaussian_blur(img, radius=2.5):
    return img.filter(ImageFilter.GaussianBlur(radius=radius))

def apply_gaussian_noise(img, std=25):
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, std, arr.shape)
    arr_noisy = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr_noisy)

def apply_brightness_shift(img, factor=0.6):
    enhancer = ImageEnhance.Brightness(img)
    return enhancer.enhance(factor)

PERTURBATIONS = {
    "clean": lambda img: img,
    "gaussian_blur": apply_gaussian_blur,
    "gaussian_noise": apply_gaussian_noise,
    "brightness_shift": apply_brightness_shift,
}

MODELS = {
    "YOLOv12s-cls (Standard)": "/content/drive/MyDrive/TCM_YOLOv12_runs/v12s_baseline/weights/best.pt",
    "YOLOv12s-cls + LTA": "/content/drive/MyDrive/TCM_YOLOv12_runs/v12s_LTA_final/weights/best.pt",
}

def evaluate_with_perturbation(weight_path, perturbation_fn, seed=42):
    np.random.seed(seed)  # 保证高斯噪声等随机扰动在不同模型间使用相同的随机种子，公平对比
    model = YOLO(weight_path)
    class_names = model.names
    name_to_idx = {v: k for k, v in class_names.items()}
    torch_model = LogitsOnlyWrapper(model.model.to(device).eval()).to(device).eval()

    test_dir = os.path.join(DATA_ROOT, "test")
    n_total, n_correct = 0, 0

    with torch.no_grad():
        for cls_name in sorted(os.listdir(test_dir)):
            cls_dir = os.path.join(test_dir, cls_name)
            if not os.path.isdir(cls_dir):
                continue
            true_idx = name_to_idx.get(cls_name)
            if true_idx is None:
                continue
            for fname in sorted(os.listdir(cls_dir)):
                if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    continue
                img_path = os.path.join(cls_dir, fname)
                try:
                    img = Image.open(img_path).convert("RGB")
                    img_perturbed = perturbation_fn(img)
                    tensor = transform(img_perturbed).unsqueeze(0).to(device)
                    pred_idx = int(torch_model(tensor).argmax(dim=1))
                    n_total += 1
                    if pred_idx == true_idx:
                        n_correct += 1
                except Exception:
                    continue

    return n_correct / n_total * 100, n_total

results_table = []

for model_name, weight_path in MODELS.items():
    if not os.path.exists(weight_path):
        print(f"⚠️ 权重文件不存在: {weight_path}，跳过 {model_name}")
        continue

    print(f"\n{'='*60}")
    print(f"评估模型: {model_name}")
    print(f"{'='*60}")

    row = {"model": model_name}
    for pert_name, pert_fn in PERTURBATIONS.items():
        acc, n = evaluate_with_perturbation(weight_path, pert_fn)
        row[pert_name] = round(acc, 2)
        print(f"  [{pert_name:<18}] Top-1 = {acc:.2f}%  (n={n})")

    row["blur_drop"] = round(row["clean"] - row["gaussian_blur"], 2)
    row["noise_drop"] = round(row["clean"] - row["gaussian_noise"], 2)
    row["brightness_drop"] = round(row["clean"] - row["brightness_shift"], 2)
    row["avg_drop"] = round((row["blur_drop"] + row["noise_drop"] + row["brightness_drop"]) / 3, 2)

    results_table.append(row)

import pandas as pd
df = pd.DataFrame(results_table)
print("\n\n" + "="*80)
print("合成鲁棒性测试汇总表")
print("="*80)
print(df.to_string(index=False))

csv_path = "/content/drive/MyDrive/TCM_YOLOv12_runs/synthetic_robustness_results.csv"
df.to_csv(csv_path, index=False)
print(f"\n结果已保存到: {csv_path}")
print("请把上面完整的汇总表发给我")


评估模型: YOLOv12s-cls (Standard)
  [clean             ] Top-1 = 82.39%  (n=636)
  [gaussian_blur     ] Top-1 = 41.51%  (n=636)
  [gaussian_noise    ] Top-1 = 53.93%  (n=636)
  [brightness_shift  ] Top-1 = 81.60%  (n=636)

评估模型: YOLOv12s-cls + LTA
  [clean             ] Top-1 = 83.81%  (n=636)
  [gaussian_blur     ] Top-1 = 36.64%  (n=636)
  [gaussian_noise    ] Top-1 = 55.97%  (n=636)
  [brightness_shift  ] Top-1 = 81.76%  (n=636)


合成鲁棒性测试汇总表
                  model  clean  gaussian_blur  gaussian_noise  brightness_shift  blur_drop  noise_drop  brightness_drop  avg_drop
YOLOv12s-cls (Standard)  82.39          41.51           53.93             81.60      40.88       28.46             0.79     23.38
     YOLOv12s-cls + LTA  83.81          36.64           55.97             81.76      47.17       27.84             2.05     25.69

结果已保存到: /content/drive/MyDrive/TCM_YOLOv12_runs/synthetic_robustness_results.csv
请把上面完整的汇总表发给我


In [10]:
"""
EfficientNet-B0 基线训练与评估脚本（步骤18）
用于与 YOLOv12s-cls (Standard) / YOLOv12s-cls + LTA 做横向对比。

数据集目录结构要求（与 Ultralytics YOLO 分类训练一致）：
    /content/dataset/
        train/<class_name>/*.jpg
        val/<class_name>/*.jpg
        test/<class_name>/*.jpg

注意：数据集在 Colab 本地磁盘 /content/dataset，不在 Google Drive。
如果本次 session 还没有解压数据集到 /content/dataset，请先运行你之前
准备数据集的那段代码（从 Drive 拷贝压缩包并解压到本地）。

用法（Colab 中直接运行整个 cell）：
    python train_efficientnet_b0_baseline.py
"""

import os
import time
import copy
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# ------------------- 配置区 -------------------
DATA_ROOT   = "/content/dataset"   # 数据集在本地磁盘，不是 Drive
SAVE_DIR    = "/content/drive/MyDrive/TCM_YOLOv12_runs/efficientnet_b0"   # 结果持久化到 Drive
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 50
LR          = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE    = 15           # 与你 YOLO 训练的 patience=15 保持一致
SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(SAVE_DIR, exist_ok=True)
torch.manual_seed(SEED)

# 数据集存在性检查，提前报错，避免和之前一样跑到 ImageFolder 才发现路径错
for split in ["train", "val", "test"]:
    split_path = os.path.join(DATA_ROOT, split)
    if not os.path.isdir(split_path):
        raise FileNotFoundError(
            f"找不到目录: {split_path}\n"
            f"请确认本次 Colab session 已经把数据集解压到 {DATA_ROOT}，"
            f"或修改脚本顶部的 DATA_ROOT 为实际路径。"
        )

# ------------------- 数据增强（对齐 augmentation study 里的 lightaug 配置） -------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=train_transform)
val_ds   = datasets.ImageFolder(os.path.join(DATA_ROOT, "val"),   transform=eval_transform)
test_ds  = datasets.ImageFolder(os.path.join(DATA_ROOT, "test"),  transform=eval_transform)

assert train_ds.classes == val_ds.classes == test_ds.classes, \
    "train/val/test 类别顺序不一致，请检查数据集目录！"

NUM_CLASSES = len(train_ds.classes)
CLASS_NAMES = train_ds.classes
print(f"类别数: {NUM_CLASSES}")
print(f"训练集: {len(train_ds)}  验证集: {len(val_ds)}  测试集: {len(test_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ------------------- 模型：EfficientNet-B0 (ImageNet 预训练) -------------------
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {n_params/1e6:.2f}M")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ------------------- 训练与验证 -------------------
def run_epoch(loader, training: bool):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


best_val_acc = 0.0
best_state = None
epochs_no_improve = 0
history = []

print("\n开始训练...\n" + "=" * 60)
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)
    scheduler.step()

    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
        "val_loss": val_loss, "val_acc": val_acc,
    })
    print(f"Epoch {epoch:03d}/{EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
          f"| val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
        torch.save(best_state, os.path.join(SAVE_DIR, "best.pt"))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\n早停触发（{PATIENCE} 轮无提升），停止于 epoch {epoch}")
            break

elapsed = time.time() - start_time
print("=" * 60)
print(f"训练完成，用时 {elapsed/60:.1f} 分钟。最佳验证准确率（用于选择 best.pt）: {best_val_acc:.4f}")

with open(os.path.join(SAVE_DIR, "train_history.json"), "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

# ------------------- 加载 best.pt，在 test 集上评估（clean baseline） -------------------
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best.pt"), map_location=DEVICE))
test_loss, test_acc = run_epoch(test_loader, training=False)

print("\n" + "=" * 60)
print("评估模型: EfficientNet-B0 (Baseline)")
print("=" * 60)
print(f"  [clean] Top-1 = {test_acc*100:.2f}%  (n={len(test_ds)})")

result_row = {
    "model": "EfficientNet-B0 (Baseline)",
    "params_M": round(n_params / 1e6, 2),
    "clean_top1": round(test_acc * 100, 2),
    "n_test": len(test_ds),
}

result_path = os.path.join(SAVE_DIR, "efficientnet_b0_baseline_result.json")
with open(result_path, "w", encoding="utf-8") as f:
    json.dump(result_row, f, ensure_ascii=False, indent=2)

print(f"\n结果已保存到: {result_path}")
print(f"权重已保存到: {os.path.join(SAVE_DIR, 'best.pt')}")
print("\n下一步：可用你现有的 synthetic robustness 评估脚本，"
      "把权重换成这里的 best.pt、模型结构换成 efficientnet_b0，"
      "对同一份 test 集做 gaussian_blur / gaussian_noise / brightness_shift 三项扰动测试，"
      "结果即可与 YOLOv12s-cls 系列拼接进同一张汇总表。")


类别数: 39
训练集: 1981  验证集: 634  测试集: 636
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 179MB/s]

模型参数量: 4.06M

开始训练...



/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 001/50 | train_loss=2.0234 train_acc=0.4649 | val_loss=1.2086 val_acc=0.6530


...

Epoch 047/50 | train_loss=0.0012 train_acc=1.0000 | val_loss=0.9520 val_acc=0.8454


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 048/50 | train_loss=0.0027 train_acc=0.9990 | val_loss=0.9667 val_acc=0.8360


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 049/50 | train_loss=0.0043 train_acc=0.9985 | val_loss=0.9727 val_acc=0.8391

早停触发（15 轮无提升），停止于 epoch 49
训练完成，用时 19.6 分钟。最佳验证准确率（用于选择 best.pt）: 0.8470

评估模型: EfficientNet-B0 (Baseline)
  [clean] Top-1 = 84.43%  (n=636)

结果已保存到: /content/drive/MyDrive/TCM_YOLOv12_runs/efficientnet_b0/efficientnet_b0_baseline_result.json
权重已保存到: /content/drive/MyDrive/TCM_YOLOv12_runs/efficientnet_b0/best.pt

下一步：可用你现有的 synthetic robustness 评估脚本，把权重换成这里的 best.pt、模型结构换成 efficientnet_b0，对同一份 test 集做 gaussian_blur / gaussian_noise / brightness_shift 三项扰动测试，结果即可与 YOLOv12s-cls 系列拼接进同一张汇总表。


In [11]:
"""
合成鲁棒性测试 —— EfficientNet-B0 (Baseline)
在同一份 test 集上评估 clean / gaussian_blur / gaussian_noise / brightness_shift
四种条件下的 Top-1 准确率，与 YOLOv12s-cls (Standard) / YOLOv12s-cls + LTA 的结果拼表对比。

扰动参数与你之前 YOLOv12s-cls 合成鲁棒性测试保持一致：
    - gaussian_blur:     PIL GaussianBlur radius=2.5
    - gaussian_noise:    高斯噪声 sigma=25 (0-255 像素值域)
    - brightness_shift:  亮度系数 0.5（整体调暗，模拟低光拍摄）

用法（Colab 中直接运行）：
    python synthetic_robustness_efficientnet_b0.py
"""

import os
import json
import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageFilter, ImageEnhance
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pandas as pd

# ------------------- 配置区 -------------------
DATA_ROOT   = "/content/dataset"
TEST_DIR    = os.path.join(DATA_ROOT, "test")
WEIGHTS_PATH = "/content/drive/MyDrive/TCM_YOLOv12_runs/efficientnet_b0/best.pt"
SAVE_DIR    = "/content/drive/MyDrive/TCM_YOLOv12_runs/efficientnet_b0"
IMG_SIZE    = 224
BATCH_SIZE  = 32
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BLUR_RADIUS       = 2.5
NOISE_SIGMA       = 25
BRIGHTNESS_FACTOR = 0.5

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

if not os.path.isdir(TEST_DIR):
    raise FileNotFoundError(f"找不到测试集目录: {TEST_DIR}，请确认本次 session 已解压数据集。")

# ------------------- 扰动函数（作用于 PIL Image，输入输出均为 PIL Image） -------------------
def apply_gaussian_blur(img: Image.Image) -> Image.Image:
    return img.filter(ImageFilter.GaussianBlur(radius=BLUR_RADIUS))

def apply_gaussian_noise(img: Image.Image) -> Image.Image:
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, NOISE_SIGMA, arr.shape)
    noisy = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy)

def apply_brightness_shift(img: Image.Image) -> Image.Image:
    enhancer = ImageEnhance.Brightness(img)
    return enhancer.enhance(BRIGHTNESS_FACTOR)

PERTURBATIONS = {
    "clean": None,
    "gaussian_blur": apply_gaussian_blur,
    "gaussian_noise": apply_gaussian_noise,
    "brightness_shift": apply_brightness_shift,
}

# ------------------- 自定义 Dataset：先扰动，再 resize/normalize -------------------
class PerturbedImageFolder(Dataset):
    def __init__(self, root, perturb_fn=None, img_size=224):
        self.samples = []
        self.classes = sorted(
            d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))
        )
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        for c in self.classes:
            class_dir = os.path.join(root, c)
            for fname in os.listdir(class_dir):
                if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    self.samples.append((os.path.join(class_dir, fname), self.class_to_idx[c]))
        self.perturb_fn = perturb_fn
        self.resize = transforms.Resize((img_size, img_size))
        self.to_tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img = self.resize(img)
        if self.perturb_fn is not None:
            img = self.perturb_fn(img)
        img = self.to_tensor(img)
        return img, label


base_ds = PerturbedImageFolder(TEST_DIR, perturb_fn=None, img_size=IMG_SIZE)
NUM_CLASSES = len(base_ds.classes)
N_TEST = len(base_ds)
print(f"类别数: {NUM_CLASSES}  测试集样本数: {N_TEST}")

# ------------------- 加载 EfficientNet-B0 + 训练好的权重 -------------------
model = models.efficientnet_b0(weights=None)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {n_params/1e6:.2f}M")

# ------------------- 评估函数 -------------------
@torch.no_grad()
def evaluate(perturb_name, perturb_fn):
    ds = PerturbedImageFolder(TEST_DIR, perturb_fn=perturb_fn, img_size=IMG_SIZE)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    top1 = correct / total * 100
    print(f"  [{perturb_name:<18}] Top-1 = {top1:.2f}%  (n={total})")
    return top1, total


print("\n" + "=" * 60)
print("评估模型: EfficientNet-B0 (Baseline)")
print("=" * 60)

results = {}
for name, fn in PERTURBATIONS.items():
    top1, n = evaluate(name, fn)
    results[name] = top1

clean = results["clean"]
blur_drop = clean - results["gaussian_blur"]
noise_drop = clean - results["gaussian_noise"]
brightness_drop = clean - results["brightness_shift"]
avg_drop = (blur_drop + noise_drop + brightness_drop) / 3

row = {
    "model": "EfficientNet-B0 (Baseline)",
    "clean": round(clean, 2),
    "gaussian_blur": round(results["gaussian_blur"], 2),
    "gaussian_noise": round(results["gaussian_noise"], 2),
    "brightness_shift": round(results["brightness_shift"], 2),
    "blur_drop": round(blur_drop, 2),
    "noise_drop": round(noise_drop, 2),
    "brightness_drop": round(brightness_drop, 2),
    "avg_drop": round(avg_drop, 2),
}

df = pd.DataFrame([row])
print("\n" + "=" * 80)
print("合成鲁棒性测试结果 —— EfficientNet-B0")
print("=" * 80)
print(df.to_string(index=False))

out_path = os.path.join(SAVE_DIR, "synthetic_robustness_efficientnet_b0.csv")
df.to_csv(out_path, index=False)
print(f"\n结果已保存到: {out_path}")

print("\n如需与 YOLOv12s-cls 系列结果合并成一张总表，可执行：")
print('  yolo_df = pd.read_csv("/content/drive/MyDrive/TCM_YOLOv12_runs/synthetic_robustness_results.csv")')
print('  combined_df = pd.concat([yolo_df, df], ignore_index=True)')

类别数: 39  测试集样本数: 636
模型参数量: 4.06M

评估模型: EfficientNet-B0 (Baseline)
  [clean             ] Top-1 = 84.43%  (n=636)
  [gaussian_blur     ] Top-1 = 22.48%  (n=636)
  [gaussian_noise    ] Top-1 = 19.18%  (n=636)
  [brightness_shift  ] Top-1 = 73.11%  (n=636)

合成鲁棒性测试结果 —— EfficientNet-B0
                     model  clean  gaussian_blur  gaussian_noise  brightness_shift  blur_drop  noise_drop  brightness_drop  avg_drop
EfficientNet-B0 (Baseline)  84.43          22.48           19.18             73.11      61.95       65.25            11.32     46.17

结果已保存到: /content/drive/MyDrive/TCM_YOLOv12_runs/efficientnet_b0/synthetic_robustness_efficientnet_b0.csv

如需与 YOLOv12s-cls 系列结果合并成一张总表，可执行：
  yolo_df = pd.read_csv("/content/drive/MyDrive/TCM_YOLOv12_runs/synthetic_robustness_results.csv")
  combined_df = pd.concat([yolo_df, df], ignore_index=True)


# ============================================================
# 【补充实验：投稿前必须完成】
#
# 目的：
# 1. 在完全相同的 YOLOv12s-cls 骨干、数据划分和训练协议下，
#    比较 Baseline / SE / CBAM / LTA。
# 2. 使用 3 个随机种子，避免单次训练偶然性。
# 3. 补齐 EfficientNet-B0 Top-5。
# 4. 导出 636 张测试图像的逐图预测。
# 5. 计算 paired bootstrap 95% CI，并自动生成论文表格。
#
# 使用方法：
# - 先完成原步骤1、2、5，确保 Drive 已挂载、数据已解压、Ultralytics 已安装。
# - 然后从“补充步骤20”开始，严格按顺序逐格运行。
# - 补充步骤22训练时间最长，可以中断后重跑；已有 best.pt 的任务会自动跳过。
# - 不要改测试集，不要使用 test 集选择模型或调整参数。
# ============================================================


In [ ]:
# ============================================================
# 补充步骤20：环境检查与统一配置
# ============================================================

import os
import sys
import json
import time
import random
import hashlib
import platform
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from ultralytics import YOLO

DATA_ROOT = "/content/dataset"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR = os.path.join(DATA_ROOT, "val")
TEST_DIR = os.path.join(DATA_ROOT, "test")

PRETRAINED_WEIGHT = "/content/yolov12s-cls.pt"
PRETRAINED_URL = "https://github.com/sunsmarterjie/yolov12/releases/download/cls/yolov12s-cls.pt"

SUPP_ROOT = "/content/drive/MyDrive/TCM_YOLOv12_runs/supplementary_attention_ablation"
PRED_DIR = os.path.join(SUPP_ROOT, "predictions")
TABLE_DIR = os.path.join(SUPP_ROOT, "tables")

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15

# 正式实验：4种结构 × 3个随机种子 = 12次训练。
# 若要先测试代码是否可运行，可临时改成 SEEDS=[42]，
# 但投稿用最终结果必须改回下面3个种子并全部跑完。
SEEDS = [42, 123, 2026]
VARIANTS = ["baseline", "SE", "CBAM", "LTA"]

for p in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if not os.path.isdir(p):
        raise FileNotFoundError(
            f"找不到 {p}。请先运行原步骤1、2，把数据集解压到 /content/dataset。"
        )

os.makedirs(SUPP_ROOT, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

# 数据包的 val/test 根目录可能还带有不属于论文39类的额外图像项。
# 原步骤4只按 train 中的39个类名统计，因此显示634/636；
# 这里先按论文固定类清单隔离额外项，再进行严格递归计数。
TARGET_CLASSES = {
    "badou", "baifuzi", "baiguo", "banxia", "beidougen", "caowu",
    "changshan", "chonglou", "chuanlianzi", "gansui", "heshi",
    "hongdaji", "huajiao", "jili", "jiulixiang", "kulianpi", "langdu",
    "liangmianzhen", "maqianzi", "mianmaguanzhong", "mubiezi",
    "naoyanghua", "qianjinzi", "qianniuzi", "shancigu", "shanglu",
    "shechuangzi", "tiannanxing", "tianxianzi", "tujingpi", "wuzhuyu",
    "xiangjiapi", "xiangsizi", "xianmao", "yadanzi", "yangjinhua",
    "yingsuqiao", "yuanhua", "zhuyazao",
}
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
QUARANTINE_ROOT = f"/content/dataset_excluded_non39_{int(time.time())}"
excluded_items = []

for split in ["train", "val", "test"]:
    split_root = os.path.join(DATA_ROOT, split)
    for item_name in sorted(os.listdir(split_root)):
        if item_name in TARGET_CLASSES:
            continue
        src = os.path.join(split_root, item_name)
        image_count = (
            sum(
                1
                for dp, _, files in os.walk(src)
                for fn in files
                if fn.lower().endswith(IMG_EXTS)
            )
            if os.path.isdir(src)
            else int(item_name.lower().endswith(IMG_EXTS))
        )
        dst_dir = os.path.join(QUARANTINE_ROOT, split)
        os.makedirs(dst_dir, exist_ok=True)
        dst = os.path.join(dst_dir, item_name)
        if os.path.exists(dst):
            dst = os.path.join(dst_dir, f"{int(time.time() * 1000)}_{item_name}")
        shutil.move(src, dst)
        excluded_items.append({
            "split": split,
            "item": item_name,
            "reason": "split根目录中的非39类项目",
            "item_type": "directory" if os.path.isdir(dst) else "file",
            "image_count": image_count,
            "quarantine_path": dst,
        })

# 继续清理39类文件夹内部的嵌套目录和隐藏图像副本。
# 论文数据结构应为 split/class/image，类别目录下面不应再有子目录。
# Colab/Jupyter 常见的 .ipynb_checkpoints 会复制图像并导致递归计数虚增。
for split in ["train", "val", "test"]:
    for cls in sorted(TARGET_CLASSES):
        class_dir = os.path.join(DATA_ROOT, split, cls)
        if not os.path.isdir(class_dir):
            continue
        for item_name in sorted(os.listdir(class_dir)):
            src = os.path.join(class_dir, item_name)
            is_nested_dir = os.path.isdir(src)
            is_hidden_image = (
                os.path.isfile(src)
                and item_name.startswith(".")
                and item_name.lower().endswith(IMG_EXTS)
            )
            if not (is_nested_dir or is_hidden_image):
                continue
            image_count = (
                sum(
                    1
                    for dp, _, files in os.walk(src)
                    for fn in files
                    if fn.lower().endswith(IMG_EXTS)
                )
                if is_nested_dir
                else 1
            )
            dst_dir = os.path.join(QUARANTINE_ROOT, split, cls)
            os.makedirs(dst_dir, exist_ok=True)
            dst = os.path.join(dst_dir, item_name)
            if os.path.exists(dst):
                dst = os.path.join(dst_dir, f"{int(time.time() * 1000)}_{item_name}")
            shutil.move(src, dst)
            excluded_items.append({
                "split": split,
                "item": f"{cls}/{item_name}",
                "reason": (
                    "类别目录内部的嵌套目录"
                    if is_nested_dir
                    else "类别目录内部的隐藏图像副本"
                ),
                "item_type": "directory" if is_nested_dir else "file",
                "image_count": image_count,
                "quarantine_path": dst,
            })

if excluded_items:
    excluded_df = pd.DataFrame(excluded_items)
    audit_path = os.path.join(TABLE_DIR, "excluded_non39_items_audit.csv")
    excluded_df.to_csv(audit_path, index=False, encoding="utf-8-sig")
    print("已隔离不属于论文39类的额外项（仅移动 /content 临时解压目录）：")
    print(excluded_df.to_string(index=False))
    print(f"隔离审计表已保存：{audit_path}")
else:
    print("未发现39类之外的额外项，无需隔离。")

# 与原步骤6保持一致：数据目录发生变化后必须删除Ultralytics旧缓存，
# 否则后续扫描可能继续沿用清理前的类别和样本索引。
removed_cache_files = []
for split in ["train", "val", "test", "test_subset"]:
    cache_path = os.path.join(DATA_ROOT, f"{split}.cache")
    if os.path.exists(cache_path):
        os.remove(cache_path)
        removed_cache_files.append(cache_path)
if removed_cache_files:
    print("已清除Ultralytics旧缓存：")
    for cache_path in removed_cache_files:
        print(f"  - {cache_path}")
else:
    print("未发现旧的Ultralytics数据缓存。")

if not os.path.exists(PRETRAINED_WEIGHT):
    rc = os.system(
        f'wget -q "{PRETRAINED_URL}" -O "{PRETRAINED_WEIGHT}"'
    )
    if rc != 0:
        raise RuntimeError("yolov12s-cls.pt 下载失败，请检查网络。")

if os.path.getsize(PRETRAINED_WEIGHT) < 1_000_000:
    raise RuntimeError("yolov12s-cls.pt 文件不完整。")

def md5sum(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

class_names = sorted(TARGET_CLASSES)

for split in ["train", "val", "test"]:
    split_root = os.path.join(DATA_ROOT, split)
    actual_classes = {
        d for d in os.listdir(split_root)
        if os.path.isdir(os.path.join(split_root, d))
    }
    if actual_classes != TARGET_CLASSES:
        missing = sorted(TARGET_CLASSES - actual_classes)
        extra = sorted(actual_classes - TARGET_CLASSES)
        raise RuntimeError(
            f"{split} 类别目录不一致。缺失={missing}，额外={extra}。请勿继续训练。"
        )

dataset_counts = {}
for split in ["train", "val", "test"]:
    root = os.path.join(DATA_ROOT, split)
    dataset_counts[split] = sum(
        1
        for cls in TARGET_CLASSES
        for dp, _, files in os.walk(os.path.join(root, cls))
        for fn in files
        if fn.lower().endswith(IMG_EXTS)
    )

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "ultralytics": __import__("ultralytics").__version__,
    "pretrained_weight": PRETRAINED_WEIGHT,
    "pretrained_weight_md5": md5sum(PRETRAINED_WEIGHT),
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "seeds": SEEDS,
    "variants": VARIANTS,
    "class_count": len(class_names),
    "dataset_counts": dataset_counts,
    "excluded_non39_items": excluded_items,
}

with open(os.path.join(SUPP_ROOT, "environment_and_protocol.json"), "w", encoding="utf-8") as f:
    json.dump(environment, f, ensure_ascii=False, indent=2)

print("=" * 72)
print("补充实验环境与数据核查结果")
print("=" * 72)
print(json.dumps(environment, ensure_ascii=False, indent=2))

if len(class_names) != 39:
    raise RuntimeError(f"类别数应为39，当前为 {len(class_names)}。")
if dataset_counts != {"train": 1981, "val": 634, "test": 636}:
    raise RuntimeError(
        "数据数量与论文不一致。应为 train/val/test = 1981/634/636，"
        f"当前为 {dataset_counts}。请勿继续训练。"
    )
print("\n✅ 补充实验环境检查通过，数据严格为39类、1981/634/636，可继续步骤21。")


In [ ]:
# ============================================================
# 补充步骤21：定义 SE、标准 CBAM、原 LTA 及统一注入逻辑
#
# 重要：
# - 三种模块插入同一层：len(backbone)-2。
# - SE/CBAM/LTA 都使用 reduction=16。
# - CBAM 使用标准通道注意力 + 7×7空间注意力。
# - LTA 严格保留原实验的零初始化方式，不追改旧结果。
# ============================================================

import torch
import torch.nn as nn


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(self.pool(x))


class CBAMChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))


class CBAMSpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(
            2, 1, kernel_size, padding=kernel_size // 2, bias=False
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))


class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = CBAMChannelAttention(channels, reduction)
        self.spatial_att = CBAMSpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        return x * self.spatial_att(x)


class LTAChannelAttention(nn.Module):
    """与原步骤10/11完全一致：最后一个1×1卷积零初始化。"""
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
        nn.init.zeros_(self.fc[2].weight)

    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))


class LTASpatialAttention(nn.Module):
    """与原步骤10/11完全一致：7×7空间卷积零初始化。"""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(
            2, 1, kernel_size, padding=kernel_size // 2, bias=False
        )
        self.sigmoid = nn.Sigmoid()
        nn.init.zeros_(self.conv.weight)

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))


class LTABlock(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = LTAChannelAttention(channels, reduction)
        self.spatial_att = LTASpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        return x * self.spatial_att(x)


class AttentionWrapper(nn.Module):
    """顶层 nn.Module，可由 PyTorch checkpoint 正常序列化。"""
    def __init__(self, orig_layer, attention):
        super().__init__()
        self.orig_layer = orig_layer
        self.attention = attention
        self.f = getattr(orig_layer, "f", -1)
        self.i = getattr(orig_layer, "i", None)
        self.type = getattr(orig_layer, "type", None)

    def forward(self, x):
        return self.attention(self.orig_layer(x))


def build_attention(kind, channels):
    if kind == "SE":
        return SEBlock(channels)
    if kind == "CBAM":
        return CBAMBlock(channels)
    if kind == "LTA":
        return LTABlock(channels)
    raise ValueError(f"未知注意力模块: {kind}")


def probe_output_channels(real_model, layer_idx, device, imgsz=224):
    target_layer = real_model.model[layer_idx]
    captured = {}

    def hook(_, __, output):
        captured["channels"] = int(output.shape[1])

    handle = target_layer.register_forward_hook(hook)
    was_training = real_model.training
    real_model.eval()
    with torch.no_grad():
        real_model(torch.zeros(1, 3, imgsz, imgsz, device=device))
    real_model.train(was_training)
    handle.remove()

    if "channels" not in captured:
        raise RuntimeError("未能动态探测目标层输出通道数。")
    return captured["channels"]


def attach_attention(real_model, kind, layer_idx, channels, device):
    original_layer = real_model.model[layer_idx]
    attention = build_attention(kind, channels).to(device)
    real_model.model[layer_idx] = AttentionWrapper(
        original_layer, attention
    ).to(device)
    return attention


def make_injection_callback(kind, status):
    def callback(trainer):
        if status["injected"]:
            return

        real_model = trainer.model
        device = next(real_model.parameters()).device
        layer_idx = len(real_model.model) - 2
        channels = probe_output_channels(
            real_model, layer_idx, device, imgsz=IMG_SIZE
        )

        attention = attach_attention(
            real_model, kind, layer_idx, channels, device
        )
        status.update({
            "injected": True,
            "layer_idx": layer_idx,
            "channels": channels,
            "module_params": sum(p.numel() for p in attention.parameters()),
        })

        if trainer.optimizer is None:
            raise RuntimeError("trainer.optimizer 不存在，新增模块无法训练。")

        param_ids_before = {
            id(p)
            for group in trainer.optimizer.param_groups
            for p in group["params"]
        }
        new_params = [
            p for p in attention.parameters()
            if id(p) not in param_ids_before
        ]
        if not new_params:
            raise RuntimeError("没有找到需要加入优化器的注意力参数。")

        decay_group = next(
            (
                g for g in trainer.optimizer.param_groups
                if float(g.get("weight_decay", 0)) > 0
            ),
            trainer.optimizer.param_groups[0],
        )
        decay_group["params"].extend(new_params)
        status["optimizer_synced"] = True

        if not (
            hasattr(trainer, "ema")
            and trainer.ema is not None
            and hasattr(trainer.ema, "ema")
        ):
            raise RuntimeError("未找到 EMA 模型，不能保证 best.pt 结构正确。")

        attach_attention(
            trainer.ema.ema, kind, layer_idx, channels, device
        )
        status["ema_synced"] = True

        total_params = sum(p.numel() for p in real_model.parameters())
        status["total_params"] = total_params
        print(
            f"[{kind}] 注入层={layer_idx}, 通道={channels}, "
            f"模块参数={status['module_params']:,}, "
            f"总参数={total_params:,}"
        )

    return callback


print("✅ SE / CBAM / LTA 与统一注入逻辑定义完成。")
print("注意：LTA 与标准 CBAM 的结构相同，当前实验主要检验原 LTA 的")
print("零初始化策略是否带来可重复优势；论文中必须如实描述这一点。")


In [ ]:
# ============================================================
# 补充步骤22：运行 4种结构 × 3个随机种子
#
# 预计运行时间较长。程序每完成一个任务就立即把结果写入 Drive。
# 若 Colab 中断，重新运行步骤20、21、22：
# - 已存在 best.pt 的任务自动跳过；
# - 未完成的任务继续训练。
# ============================================================

import gc
import os
import json
import random
import shutil
import time
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO


def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# 独立硬门禁：即使跳过步骤20，步骤22也绝不能在错误数据上启动。
expected_counts = {"train": 1981, "val": 634, "test": 636}
step22_counts = {}
step22_class_sets = {}
for split in ["train", "val", "test"]:
    split_root = os.path.join(DATA_ROOT, split)
    if not os.path.isdir(split_root):
        raise RuntimeError(f"步骤22终止：找不到 {split_root}，请先运行步骤20。")
    step22_class_sets[split] = {
        d for d in os.listdir(split_root)
        if os.path.isdir(os.path.join(split_root, d))
    }
    step22_counts[split] = sum(
        1
        for cls in step22_class_sets[split]
        for dp, _, files in os.walk(os.path.join(split_root, cls))
        for fn in files
        if fn.lower().endswith(IMG_EXTS)
    )

bad_class_sets = {
    split: {
        "missing": sorted(TARGET_CLASSES - classes),
        "extra": sorted(classes - TARGET_CLASSES),
    }
    for split, classes in step22_class_sets.items()
    if classes != TARGET_CLASSES
}
if bad_class_sets or step22_counts != expected_counts:
    raise RuntimeError(
        "步骤22已硬停止，禁止在错误数据上训练。"
        f"类别差异={bad_class_sets}；数量={step22_counts}；"
        f"应为={expected_counts}。请先重新运行最新版步骤20。"
    )
print("✅ 步骤22二次数据门禁通过：39类，train/val/test=1981/634/636。")


task_rows = []

for variant in VARIANTS:
    for seed in SEEDS:
        run_name = f"v12s_{variant.lower()}_seed{seed}"
        run_dir = os.path.join(SUPP_ROOT, run_name)
        best_path = os.path.join(run_dir, "weights", "best.pt")
        status_path = os.path.join(run_dir, "supplement_status.json")

        print("\n" + "=" * 78)
        print(f"任务: {variant} | seed={seed}")
        print("=" * 78)

        completed_status = {}
        if os.path.exists(status_path):
            try:
                with open(status_path, "r", encoding="utf-8") as f:
                    completed_status = json.load(f)
            except Exception:
                completed_status = {}

        is_verified_complete = (
            os.path.exists(best_path)
            and completed_status.get("status") == "completed"
            and completed_status.get("variant") == variant
            and completed_status.get("seed") == seed
        )
        if is_verified_complete:
            print(f"✅ 已有完整完成标记，跳过训练: {best_path}")
            task_rows.append({
                "variant": variant,
                "seed": seed,
                "status": "verified_completed_checkpoint",
                "best_path": best_path,
            })
            continue

        if os.path.isdir(run_dir):
            preserved_dir = (
                run_dir + f"_INVALID_OR_INCOMPLETE_{int(time.time())}"
            )
            shutil.move(run_dir, preserved_dir)
            print(f"⚠️ 发现无完整标记的旧任务，已保留并移出正式目录: {preserved_dir}")

        set_all_seeds(seed)
        model = YOLO(PRETRAINED_WEIGHT)
        injection_status = {
            "variant": variant,
            "seed": seed,
            "injected": variant == "baseline",
            "optimizer_synced": variant == "baseline",
            "ema_synced": variant == "baseline",
        }

        if variant != "baseline":
            model.add_callback(
                "on_pretrain_routine_end",
                make_injection_callback(variant, injection_status),
            )

        try:
            model.train(
                data=DATA_ROOT,
                epochs=EPOCHS,
                imgsz=IMG_SIZE,
                batch=BATCH_SIZE,
                patience=PATIENCE,
                seed=seed,
                deterministic=True,
                project=SUPP_ROOT,
                name=run_name,
                exist_ok=True,
                plots=True,
                verbose=True,
            )

            if not os.path.exists(best_path):
                raise RuntimeError(f"训练结束但未找到 {best_path}")

            if variant != "baseline" and not all(
                injection_status.get(k, False)
                for k in ["injected", "optimizer_synced", "ema_synced"]
            ):
                raise RuntimeError(
                    f"{variant} 注入自检失败: {injection_status}"
                )

            injection_status["status"] = "completed"
            injection_status["best_path"] = best_path
            injection_status["dataset_counts"] = step22_counts
            injection_status["pretrained_weight_md5"] = md5sum(PRETRAINED_WEIGHT)
            with open(status_path, "w", encoding="utf-8") as f:
                json.dump(injection_status, f, ensure_ascii=False, indent=2)

            task_rows.append({
                "variant": variant,
                "seed": seed,
                "status": "completed",
                "best_path": best_path,
            })
            print(f"✅ 完成并保存: {best_path}")

        except Exception as e:
            injection_status["status"] = "failed"
            injection_status["error"] = repr(e)
            os.makedirs(run_dir, exist_ok=True)
            with open(status_path, "w", encoding="utf-8") as f:
                json.dump(injection_status, f, ensure_ascii=False, indent=2)
            task_rows.append({
                "variant": variant,
                "seed": seed,
                "status": "failed",
                "best_path": None,
                "error": repr(e),
            })
            pd.DataFrame(task_rows).to_csv(
                os.path.join(TABLE_DIR, "training_task_status.csv"),
                index=False,
            )
            raise
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        pd.DataFrame(task_rows).to_csv(
            os.path.join(TABLE_DIR, "training_task_status.csv"),
            index=False,
        )

task_df = pd.DataFrame(task_rows)
print("\n" + "=" * 78)
print("全部训练任务状态")
print("=" * 78)
print(task_df.to_string(index=False))

missing = []
for variant in VARIANTS:
    for seed in SEEDS:
        run_dir = os.path.join(
            SUPP_ROOT,
            f"v12s_{variant.lower()}_seed{seed}",
        )
        p = os.path.join(run_dir, "weights", "best.pt")
        status_p = os.path.join(run_dir, "supplement_status.json")
        completed = {}
        if os.path.exists(status_p):
            try:
                with open(status_p, "r", encoding="utf-8") as f:
                    completed = json.load(f)
            except Exception:
                completed = {}
        if not (os.path.exists(p) and completed.get("status") == "completed"):
            missing.append(run_dir)

if missing:
    print("\n⚠️ 尚未完成以下任务，请再次运行本步骤：")
    print("\n".join(missing))
else:
    print("\n✅ 12个 checkpoint 全部完成，可以运行补充步骤23。")


In [ ]:
# ============================================================
# 补充步骤23：逐图评估全部 YOLO 模型
#
# 输出：
# - 每个模型/种子一份 predictions CSV（636行）
# - attention_ablation_metrics.csv
# - 包含 Top-1、Top-5、参数量、GFLOPs、平均推理时间
# ============================================================

import gc
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
test_paths = sorted(
    str(p)
    for p in Path(TEST_DIR).glob("*/*")
    if p.suffix.lower() in IMG_EXTS
)
if len(test_paths) != 636:
    raise RuntimeError(f"测试集应有636张图，当前找到 {len(test_paths)} 张。")


def names_to_dict(names):
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    return {i: str(v) for i, v in enumerate(names)}


metric_rows = []

for variant in VARIANTS:
    for seed in SEEDS:
        run_name = f"v12s_{variant.lower()}_seed{seed}"
        best_path = os.path.join(
            SUPP_ROOT, run_name, "weights", "best.pt"
        )
        pred_path = os.path.join(
            PRED_DIR, f"{run_name}_test_predictions.csv"
        )

        print("\n" + "=" * 78)
        print(f"逐图评估: {variant} | seed={seed}")
        print("=" * 78)

        if not os.path.exists(best_path):
            raise FileNotFoundError(
                f"缺少 {best_path}，请先完成补充步骤22。"
            )

        model = YOLO(best_path)
        names = names_to_dict(model.names)
        name_to_idx = {v: k for k, v in names.items()}

        unknown_classes = sorted(
            {Path(p).parent.name for p in test_paths} - set(name_to_idx)
        )
        if unknown_classes:
            raise RuntimeError(
                f"checkpoint 类别名与测试目录不一致: {unknown_classes}"
            )

        n_params = sum(p.numel() for p in model.model.parameters())
        try:
            from ultralytics.utils.torch_utils import get_flops
            gflops = float(get_flops(model.model, imgsz=IMG_SIZE))
        except Exception as e:
            print(f"⚠️ GFLOPs 计算失败，将记为空值: {e}")
            gflops = np.nan

        rows = []
        inference_ms = []
        stream = model.predict(
            source=test_paths,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            device=0 if torch.cuda.is_available() else "cpu",
            stream=True,
            verbose=False,
        )

        for expected_path, result in zip(test_paths, stream):
            true_name = Path(expected_path).parent.name
            true_idx = int(name_to_idx[true_name])
            top1_idx = int(result.probs.top1)
            top5_idx = [int(x) for x in result.probs.top5]
            conf = result.probs.data.detach().cpu().numpy()
            inference_ms.append(float(result.speed.get("inference", np.nan)))
            rows.append({
                "relative_path": str(
                    Path(expected_path).relative_to(TEST_DIR)
                ),
                "true_idx": true_idx,
                "true_class": true_name,
                "pred_idx": top1_idx,
                "pred_class": names[top1_idx],
                "top1_confidence": float(conf[top1_idx]),
                "top1_correct": int(top1_idx == true_idx),
                "top5_indices": "|".join(map(str, top5_idx)),
                "top5_correct": int(true_idx in top5_idx),
            })

        pred_df = pd.DataFrame(rows)
        if len(pred_df) != 636:
            raise RuntimeError(
                f"{run_name} 只生成了 {len(pred_df)} 条预测，应为636。"
            )
        if pred_df["relative_path"].duplicated().any():
            raise RuntimeError(f"{run_name} 出现重复图片路径。")

        pred_df.to_csv(pred_path, index=False)
        top1 = 100.0 * pred_df["top1_correct"].mean()
        top5 = 100.0 * pred_df["top5_correct"].mean()
        mean_ms = float(np.nanmean(inference_ms))

        metric_rows.append({
            "variant": variant,
            "seed": seed,
            "top1_pct": top1,
            "top5_pct": top5,
            "params": n_params,
            "gflops": gflops,
            "mean_inference_ms_per_image": mean_ms,
            "n_test": len(pred_df),
            "checkpoint": best_path,
            "prediction_csv": pred_path,
        })
        pd.DataFrame(metric_rows).to_csv(
            os.path.join(TABLE_DIR, "attention_ablation_metrics.csv"),
            index=False,
        )
        print(
            f"Top-1={top1:.2f}% | Top-5={top5:.2f}% | "
            f"Params={n_params:,} | GFLOPs={gflops:.3f} | "
            f"Inference={mean_ms:.3f} ms/image"
        )

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

metrics_df = pd.DataFrame(metric_rows)
print("\n✅ YOLO逐图评估完成")
print(metrics_df.to_string(index=False))


In [ ]:
# ============================================================
# 补充步骤24：补齐 EfficientNet-B0 Top-5 与逐图预测
#
# 此步骤不重新训练 EfficientNet，只读取原步骤18生成的 best.pt。
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

EFF_WEIGHTS = (
    "/content/drive/MyDrive/TCM_YOLOv12_runs/"
    "efficientnet_b0/best.pt"
)
EFF_PRED_CSV = os.path.join(
    PRED_DIR, "efficientnet_b0_test_predictions.csv"
)

if not os.path.exists(EFF_WEIGHTS):
    raise FileNotFoundError(
        f"找不到 {EFF_WEIGHTS}，请先运行原步骤18完成 EfficientNet-B0 训练。"
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225],
    ),
])
test_ds = datasets.ImageFolder(TEST_DIR, transform=eval_transform)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

eff_model = models.efficientnet_b0(weights=None)
in_features = eff_model.classifier[1].in_features
eff_model.classifier[1] = nn.Linear(in_features, len(test_ds.classes))
eff_model.load_state_dict(torch.load(EFF_WEIGHTS, map_location=device))
eff_model = eff_model.to(device)
eff_model.eval()

rows = []
sample_offset = 0
elapsed_ms = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        logits = eff_model(images)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed_ms.extend([
            (time.perf_counter() - t0) * 1000.0 / images.size(0)
        ] * images.size(0))

        probs = torch.softmax(logits, dim=1)
        top5_prob, top5_idx = probs.topk(5, dim=1)
        top1_idx = top5_idx[:, 0]

        for j in range(images.size(0)):
            dataset_path, dataset_true = test_ds.samples[sample_offset + j]
            true_idx = int(labels[j].item())
            if true_idx != int(dataset_true):
                raise RuntimeError("ImageFolder 样本顺序与 DataLoader 标签不一致。")
            pred_idx = int(top1_idx[j].item())
            indices = [int(x) for x in top5_idx[j].cpu().tolist()]
            rows.append({
                "relative_path": str(
                    Path(dataset_path).relative_to(TEST_DIR)
                ),
                "true_idx": true_idx,
                "true_class": test_ds.classes[true_idx],
                "pred_idx": pred_idx,
                "pred_class": test_ds.classes[pred_idx],
                "top1_confidence": float(top5_prob[j, 0].item()),
                "top1_correct": int(pred_idx == true_idx),
                "top5_indices": "|".join(map(str, indices)),
                "top5_correct": int(true_idx in indices),
            })
        sample_offset += images.size(0)

eff_pred_df = pd.DataFrame(rows)
if len(eff_pred_df) != 636:
    raise RuntimeError(
        f"EfficientNet 预测数为 {len(eff_pred_df)}，应为636。"
    )
eff_pred_df.to_csv(EFF_PRED_CSV, index=False)

eff_top1 = 100.0 * eff_pred_df["top1_correct"].mean()
eff_top5 = 100.0 * eff_pred_df["top5_correct"].mean()
eff_params = sum(p.numel() for p in eff_model.parameters())

eff_row = pd.DataFrame([{
    "variant": "EfficientNet-B0",
    "seed": 42,
    "top1_pct": eff_top1,
    "top5_pct": eff_top5,
    "params": eff_params,
    "gflops": np.nan,
    "mean_inference_ms_per_image": float(np.mean(elapsed_ms)),
    "n_test": len(eff_pred_df),
    "checkpoint": EFF_WEIGHTS,
    "prediction_csv": EFF_PRED_CSV,
}])
eff_row.to_csv(
    os.path.join(TABLE_DIR, "efficientnet_b0_metrics_complete.csv"),
    index=False,
)

print("=" * 72)
print("EfficientNet-B0 测试集结果")
print("=" * 72)
print(f"Top-1: {eff_top1:.2f}%")
print(f"Top-5: {eff_top5:.2f}%")
print(f"参数量: {eff_params:,}")
print(f"逐图预测: {EFF_PRED_CSV}")


In [ ]:
# ============================================================
# 补充步骤25：paired bootstrap 95% CI 与多种子统计
#
# 统计原则：
# - 单模型准确率 CI：对636张测试图像有放回抽样。
# - 模型差值 CI：同一次抽样使用相同图片索引（paired bootstrap）。
# - 不使用 test 集调参；这里只对最终 checkpoint 做统计推断。
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd

N_BOOT = 10000
BOOTSTRAP_SEED = 20260827
rng = np.random.default_rng(BOOTSTRAP_SEED)


def load_correctness(csv_path):
    df = pd.read_csv(csv_path).sort_values("relative_path").reset_index(drop=True)
    if len(df) != 636:
        raise RuntimeError(f"{csv_path} 的行数不是636。")
    if df["relative_path"].duplicated().any():
        raise RuntimeError(f"{csv_path} 有重复路径。")
    return df["relative_path"], df["top1_correct"].to_numpy(dtype=np.int8)


def bootstrap_accuracy(correct, n_boot=N_BOOT):
    n = len(correct)
    values = np.empty(n_boot, dtype=np.float64)
    for start in range(0, n_boot, 500):
        stop = min(start + 500, n_boot)
        idx = rng.integers(0, n, size=(stop - start, n))
        values[start:stop] = correct[idx].mean(axis=1) * 100.0
    return (
        float(correct.mean() * 100.0),
        float(np.percentile(values, 2.5)),
        float(np.percentile(values, 97.5)),
    )


def paired_bootstrap(a, b, n_boot=N_BOOT):
    """返回 a-b 的点估计、95% CI 和双侧 bootstrap p值。"""
    if len(a) != len(b):
        raise ValueError("paired bootstrap 要求样本数相同。")
    diff = a.astype(np.float64) - b.astype(np.float64)
    n = len(diff)
    values = np.empty(n_boot, dtype=np.float64)
    for start in range(0, n_boot, 500):
        stop = min(start + 500, n_boot)
        idx = rng.integers(0, n, size=(stop - start, n))
        values[start:stop] = diff[idx].mean(axis=1) * 100.0
    point = float(diff.mean() * 100.0)
    lo, hi = np.percentile(values, [2.5, 97.5])
    p_two_sided = float(
        min(
            1.0,
            2.0 * min(
                np.mean(values <= 0.0),
                np.mean(values >= 0.0),
            ),
        )
    )
    return point, float(lo), float(hi), p_two_sided


metric_path = os.path.join(TABLE_DIR, "attention_ablation_metrics.csv")
metrics_df = pd.read_csv(metric_path)

ci_rows = []
pair_rows = []

for _, row in metrics_df.iterrows():
    paths, correct = load_correctness(row["prediction_csv"])
    acc, lo, hi = bootstrap_accuracy(correct)
    ci_rows.append({
        "variant": row["variant"],
        "seed": int(row["seed"]),
        "top1_pct": acc,
        "top1_ci95_low": lo,
        "top1_ci95_high": hi,
        "n_test": len(correct),
    })

for seed in SEEDS:
    base_row = metrics_df[
        (metrics_df["variant"] == "baseline")
        & (metrics_df["seed"] == seed)
    ].iloc[0]
    base_paths, base_correct = load_correctness(
        base_row["prediction_csv"]
    )

    for variant in ["SE", "CBAM", "LTA"]:
        comp_row = metrics_df[
            (metrics_df["variant"] == variant)
            & (metrics_df["seed"] == seed)
        ].iloc[0]
        comp_paths, comp_correct = load_correctness(
            comp_row["prediction_csv"]
        )
        if not base_paths.equals(comp_paths):
            raise RuntimeError(
                f"seed={seed} 时 baseline 与 {variant} 图片顺序不一致。"
            )

        delta, lo, hi, p_boot = paired_bootstrap(
            comp_correct, base_correct
        )
        pair_rows.append({
            "comparison": f"{variant} - baseline",
            "seed": seed,
            "delta_top1_pp": delta,
            "delta_ci95_low_pp": lo,
            "delta_ci95_high_pp": hi,
            "bootstrap_p_two_sided": p_boot,
            "ci_excludes_zero": bool(lo > 0 or hi < 0),
        })

ci_df = pd.DataFrame(ci_rows)
pair_df = pd.DataFrame(pair_rows)

seed_summary = (
    metrics_df.groupby("variant", as_index=False)
    .agg(
        top1_mean_pct=("top1_pct", "mean"),
        top1_sd_pct=("top1_pct", "std"),
        top5_mean_pct=("top5_pct", "mean"),
        top5_sd_pct=("top5_pct", "std"),
        params=("params", "first"),
        gflops=("gflops", "first"),
        inference_ms_mean=("mean_inference_ms_per_image", "mean"),
    )
)

delta_seed_summary = (
    pair_df.groupby("comparison", as_index=False)
    .agg(
        delta_mean_pp=("delta_top1_pp", "mean"),
        delta_sd_pp=("delta_top1_pp", "std"),
        seeds=("seed", "count"),
    )
)

ci_df.to_csv(
    os.path.join(TABLE_DIR, "bootstrap_model_accuracy_ci.csv"),
    index=False,
)
pair_df.to_csv(
    os.path.join(TABLE_DIR, "bootstrap_paired_differences.csv"),
    index=False,
)
seed_summary.to_csv(
    os.path.join(TABLE_DIR, "multiseed_summary.csv"),
    index=False,
)
delta_seed_summary.to_csv(
    os.path.join(TABLE_DIR, "multiseed_delta_summary.csv"),
    index=False,
)

print("\n" + "=" * 80)
print("多随机种子汇总（论文主表候选）")
print("=" * 80)
print(seed_summary.to_string(index=False))

print("\n" + "=" * 80)
print("逐种子 paired bootstrap：注意力模块 - baseline")
print("=" * 80)
print(pair_df.to_string(index=False))

print("\n" + "=" * 80)
print("跨种子差值汇总")
print("=" * 80)
print(delta_seed_summary.to_string(index=False))


In [ ]:
# ============================================================
# 补充步骤26：完整性检查与一键打包
#
# 运行成功后，把生成的 ZIP 文件下载并发给我。
# 我会据此决定：
# A. LTA方法论文路线；或
# B. 注意力机制/架构系统评测论文路线。
# ============================================================

import json
import os
import shutil
from pathlib import Path

required_files = [
    os.path.join(SUPP_ROOT, "environment_and_protocol.json"),
    os.path.join(TABLE_DIR, "attention_ablation_metrics.csv"),
    os.path.join(TABLE_DIR, "efficientnet_b0_metrics_complete.csv"),
    os.path.join(TABLE_DIR, "bootstrap_model_accuracy_ci.csv"),
    os.path.join(TABLE_DIR, "bootstrap_paired_differences.csv"),
    os.path.join(TABLE_DIR, "multiseed_summary.csv"),
    os.path.join(TABLE_DIR, "multiseed_delta_summary.csv"),
]

for variant in VARIANTS:
    for seed in SEEDS:
        required_files.append(
            os.path.join(
                PRED_DIR,
                f"v12s_{variant.lower()}_seed{seed}_test_predictions.csv",
            )
        )
required_files.append(
    os.path.join(PRED_DIR, "efficientnet_b0_test_predictions.csv")
)

missing = [p for p in required_files if not os.path.exists(p)]
if missing:
    print("❌ 结果不完整，缺少以下文件：")
    print("\n".join(missing))
    raise RuntimeError("请完成缺失步骤后再打包。")

manifest = []
for p in required_files:
    manifest.append({
        "path": p,
        "size_bytes": os.path.getsize(p),
        "md5": md5sum(p),
    })

manifest_path = os.path.join(SUPP_ROOT, "result_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

zip_base = (
    "/content/drive/MyDrive/"
    "TCM_supplementary_experiment_results"
)
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir=SUPP_ROOT,
)

print("=" * 80)
print("✅ 所有补充实验结果完整")
print("=" * 80)
print(f"结果目录: {SUPP_ROOT}")
print(f"打包文件: {zip_path}")
print("\n请把这个 ZIP 文件发给我，不要只发截图。")
